# MLflow Part 2


🎯 **本章學完你將能學會什麼：**

- 掌握 MLflow CallbackHandler 的使用方式，將 LangChain 的輸入/輸出與模型紀錄自動化  
- 理解 Autolog 模式下的自動追蹤原理與應用場景  
- 能夠在 MLflow UI 中觀察 Langchain 的運作過程、輸入輸出與執行時間  

📘 **最終你將具備的能力：**  
能夠建立可追蹤的 LLM 實驗環境，將 LangChain 與 MLflow 整合，為模型開發、除錯與實驗管理奠定基礎。  


mlflow server --host 127.0.0.1 --port 8080

## 紀錄內容

In [1]:
import os

os.chdir("../../")

In [2]:
import mlflow
from textwrap import dedent
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_community.callbacks import MlflowCallbackHandler
from langchain_openai import ChatOpenAI

from initialization import credential_init

def build_standard_chat_prompt_template(kwargs):

    messages = []
 
    if 'system' in kwargs:
        content = kwargs.get('system')
        prompt = PromptTemplate(**content)
        message = SystemMessagePromptTemplate(prompt=prompt)
        messages.append(message)  

    if 'human' in kwargs:
        content = kwargs.get('human')
        prompt = PromptTemplate(**content)
        message = HumanMessagePromptTemplate(prompt=prompt)
        messages.append(message)
        
    chat_prompt_template = ChatPromptTemplate.from_messages(messages)
    
    return chat_prompt_template

experiment = "week_4"
tracking_url = "http://127.0.0.1:8080"

credential_init()

C:\Users\Ling\miniconda3\envs\aicg\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
mlflow.set_tracking_uri(uri=tracking_url)

# Start or get an MLflow run explicitly
mlflow.set_experiment(experiment)

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1774626938009, experiment_id='1', last_update_time=1774626938009, lifecycle_stage='active', name='week_4', tags={}, workspace='default'>

### MlflowCallbackHandler

追蹤並記錄語言模型的輸入和輸出

In [4]:
with mlflow.start_run(run_name="my-llm-run") as run:
# run = mlflow.start_run(run_name="my-llm-run")
    run_id = run.info.run_id
    print(f"Using run_id={run_id}")

    # Attach the run_id so all logs go into this run
    mlflow_cb = MlflowCallbackHandler(
        experiment=experiment,
        run_id=run_id,
        tracking_uri=tracking_url,
    )

    model = ChatOpenAI(
        model_name="gpt-4o-mini",
        temperature=0,
        callbacks=[mlflow_cb]
    )

    prompt = PromptTemplate(
        input_variables=["product"],
        template="What is a good name for a company that makes {product}? Response in traditional Chinese (繁體中文)",
    )

    chain = prompt|model

    # First call logs into this run
    chain.invoke({"product": "陽電子攻城炮"})

    # Second call also logs into the SAME run_id
    chain.invoke({"product": "旋風魚雷 (Warhammer 40k, Exterminatus)"})

    chain.invoke({"product": "人形MS/Gundam"})
    
# Finally flush once
mlflow_cb.flush_tracker()

Could not import spacy python package. Please install it with `pip install spacy`.
Could not import textstat python package. Please install it with `pip install textstat`.


Using run_id=8693f3a38c32437a87cf50bffa6d2102
🏃 View run my-llm-run at: http://127.0.0.1:8080/#/experiments/1/runs/8693f3a38c32437a87cf50bffa6d2102
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


In [5]:
run_id = run_id
artifact_path = "table_session_analysis.html"   # artifacts 內的相對路徑位置

# Download to a local directory
local_dir = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path=artifact_path,
                                                dst_path="tutorial/week_4", 
                                                tracking_uri=tracking_url,
                                                )

print("Downloaded to:", local_dir)

Downloaded to: C:\Users\Ling\Dev\RookieSavior_LLM_Applications\tutorial\week_4\table_session_analysis.html


In [6]:
# !pip install lxml

In [7]:
import pandas as pd

# 記得要加上encoding='utf-8',否則中文會變成亂碼
df = pd.read_html("tutorial/week_4/table_session_analysis.html", encoding='utf-8')

In [8]:
df[0]

,Unnamed: 0,prompt_step,prompt,name,output_step,output,token_usage_total_tokens,token_usage_prompt_tokens,token_usage_completion_tokens
0,0,1,Human: What is a good name for a company that ...,ChatOpenAI,2,一個適合製造陽電子攻城炮的公司的名稱可以是「陽光科技有限公司」。這個名字不僅突顯了陽電子的特...,84,33,51
1,1,3,Human: What is a good name for a company that ...,ChatOpenAI,4,一個適合製造旋風魚雷的公司名稱可以是「滅絕科技有限公司」。這個名稱強調了產品的強大和致命性，...,94,42,52
2,2,5,Human: What is a good name for a company that ...,ChatOpenAI,6,一個適合製作人形MS/Gundam的公司的名稱可以是「機甲工坊」或「鋼彈創意」。這些名稱都能...,84,33,51


In [9]:
print(df[0].iloc[0])

Unnamed: 0                                                                       0
prompt_step                                                                      1
prompt                           Human: What is a good name for a company that ...
name                                                                    ChatOpenAI
output_step                                                                      2
output                           一個適合製造陽電子攻城炮的公司的名稱可以是「陽光科技有限公司」。這個名字不僅突顯了陽電子的特...
token_usage_total_tokens                                                        84
token_usage_prompt_tokens                                                       33
token_usage_completion_tokens                                                   51
Name: 0, dtype: object


## Autolog

此模式完全不會寫入任何 JSON 檔案 —— 相反地，它會將你的 LangChain 執行過程（traces/spans）捕捉並記錄到 MLflow 的實驗追蹤與追蹤（tracing）介面中。這表示你可以在 MLflow 的使用者介面中看到輸入/輸出、執行時間以及巢狀結構，而不是以 .json 檔案的形式儲存。

In [10]:
# run_id

In [11]:
# Enable autologging — this instruments LangChain automatically
mlflow.langchain.autolog()

model = ChatOpenAI(
    model_name="gpt-4o-mini",
    temperature=0,
    name="hello model"
)

prompt = PromptTemplate(
    input_variables=["product"],
    template="What is a good name in traditional Chinese for a company that makes {product}?",
)

chain = prompt|model

# Run the chain
mlflow.set_tracking_uri(uri=tracking_url)

# Create a new MLflow Experiment
mlflow.set_experiment(experiment)

with mlflow.start_run(run_name="autolog_first_test") as run:
    chain.invoke({"product": "光茅 (戰槌40k)"})
    chain.invoke({"product": "旋風魚雷 (Warhammer 40k, Exterminatus)"})
    chain.invoke({"product": "人形MS/Gundam"})

🏃 View run autolog_first_test at: http://127.0.0.1:8080/#/experiments/1/runs/f8cf830cb6e44355888ab5249adb0848
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


[Trace(trace_id=tr-7c866413853c9872ae8edffac6d55b82), Trace(trace_id=tr-0f578c187777b8d3935cc040a3327751), Trace(trace_id=tr-a8f4b62e569d603526d69f95be051ae3)]

## Reflection + Revision Pipeline

一個模型不夠，你可以用兩個。上面兩個範例都只用了一個模型，還沒外加RAG之類的

🎯 **本章學完你將能學會什麼：**

- 理解如何利用多個模型建立反饋（Reflection）與修訂（Revision）流程  
- 學會設計結構化輸出（PydanticOutputParser）以生成可解析的結果  
- 掌握 RunnablePassthrough 與 RunnableLambda 的應用，打造可組合的 LangChain Pipeline  
- 理解如何在 MLflow 中記錄多階段模型的追蹤數據  

📘 **最終你將具備的能力：**  
能夠構建具備反思與修訂能力的複合式 LLM Pipeline，並在 MLflow 內完整追蹤其運行歷程。  

### Reflection

使用某年的學測/指考的作文作為範例

In [12]:
query = dedent("""
俗話說：「龍生九子，各有不同。」在廣闊浩瀚的海洋之中，就有一頭孤獨的鯨魚——五十二赫茲鯨魚。牠聲音的頻率天生便比同伴還要高，這項特別之處，也導致了牠與同伴產生了無法溝通的鴻溝。看見這則故事的我，不禁思考，在如此多元的人間，是否也有像五十二赫茲鯨魚一般，天生便與眾不同？

回首童年，我印象最為深刻的一刻，是初識字時，與文字互相理解的那一瞬、是當我第一次讀完一個句子時，它將自身的意義傳入我腦中的那一瞬。自此，我便對文字、語言抱有特殊的感情，也十分享受閱讀與朗誦。那種將自身與文字經由一點一滴積累而連接起來的感情，使我心靈感到十分富足。

而當我步入校園接觸同儕時，驀然驚覺我與別人的閱讀速度十分不同。每當我已讀完一篇文章，但同學可能只完成了一半甚至三分之二。同時，我在生字讀音方面也異常的執著，因此被同學抱怨有「文字潔癖」。面對同儕抱怨的我，也只好強忍對耳邊時而出現字錯讀音的不適，開始刻意忽略心裡對它的執念，只為想要與別人一樣，想要和朋友互相理解。

直到多年前，因緣際會之下認識了「五十二赫茲」這獨特的存在。牠的身影在我心中烙下一道深刻的痕跡。因為牠，我開始接受自己與他人的不同；也因為牠，我明白了，我對文字的執著，並不是一種負面的特質，而是上天賜予我的禮物，我開始在寫作上揮灑自如。這讓我知道，不要在一開始便用否定的眼光看待自己的特質。也許這特別之處，會使我們與五十二赫茲鯨魚一般孤獨，會使我們遭受他人的不理解與排斥，但也會讓我們與眾不同。

關於此，我想說的是，勇敢地綻放自己的特別，也讓自己成為自己和世人眼中，最閃耀的五十二赫茲鯨魚。
""")


def create_feedback_pipeline(mlflow_callback):

    ## Teacher LLM
    system_template = dedent("""
    你是一個教學與寫作經驗豐富的台灣大學中文系教授，你要來負責給予作文評分與回饋。
    """)
    
    human_template = dedent("""
    Title: {title}
    
    Article:
    {article}
    """)
    
    model = ChatOpenAI(openai_api_key=os.environ['OPENAI_API_KEY'],
                       model_name="gpt-4o-mini", temperature=0,
                       callbacks=[mlflow_callback],
                       name="feedback_model")
    
    input_ = {"system": {"template": system_template},
              "human": {"template": human_template,
                        "input_variable": ["title", "article"],
                        }}
    
    chat_prompt_template = build_standard_chat_prompt_template(input_)
    
    feedback_pipeline = chat_prompt_template|model|StrOutputParser()

    return feedback_pipeline

In [13]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser


class Output(BaseModel):
    name: str = Field(description="The revised article in traditional Chinese (繁體中文), please do not include the title.")


output_parser = PydanticOutputParser(pydantic_object=Output)
format_instructions = output_parser.get_format_instructions()


def create_revision_pipeline(mlflow_callback):
    ## Generate
    system_template = dedent("""
    你是一個在準備考試的高中生，你將根據反饋強化的作文內容。
    """)
    
    human_template = dedent("""
    Title: {title}
    
    Old Article:
    {article}
    
    Feedback:
    {feedback}

    Output format instructions: {format_instructions}
    
    Revised Article:
    """)
    
    input_ = {"system": {"template": system_template},
              "human": {"template": human_template,
                        "input_variable": ["title", "article", "feedback"],
                        "partial_variables": {'format_instructions': format_instructions}}}
    
    chat_prompt_template = build_standard_chat_prompt_template(input_)

    model = ChatOpenAI(openai_api_key=os.environ['OPENAI_API_KEY'],
                       model_name="gpt-4o-mini", temperature=0,
                       callbacks=[mlflow_callback],
                       name='revision_model')
    
    revision_pipeline = chat_prompt_template|model|output_parser

    return revision_pipeline

在呼叫MLflow後，進行 Reflection -> Revision

In [14]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

run_name = "Reflection_Revision"

mlflow.set_tracking_uri(uri=tracking_url)

# Start or get an MLflow run explicitly
mlflow.set_experiment(experiment)
with mlflow.start_run(run_name=run_name) as run:
    run_id = run.info.run_id
    print(f"Using run_id={run_id}")

    # Attach the run_id so all logs go into this run
    mlflow_cb = MlflowCallbackHandler(
        experiment=experiment,
        run_id=run_id,
        tracking_uri=tracking_url,
    )

    feedback_pipeline = create_feedback_pipeline(mlflow_callback=mlflow_cb)
    revision_pipeline = create_revision_pipeline(mlflow_callback=mlflow_cb)
    
    whole_pipeline = RunnablePassthrough.assign(feedback=feedback_pipeline)|revision_pipeline|RunnableLambda(lambda x: x.name)

    result = whole_pipeline.invoke({"article": query,
                                    "title": "關於五十二赫茲，我想說的是…"}
                                  )
    
    
# Finally flush once
mlflow_cb.flush_tracker()

Could not import spacy python package. Please install it with `pip install spacy`.
Could not import textstat python package. Please install it with `pip install textstat`.


Using run_id=5845452695c64d1f884eda218cdd7e0c
🏃 View run Reflection_Revision at: http://127.0.0.1:8080/#/experiments/1/runs/5845452695c64d1f884eda218cdd7e0c
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


Trace(trace_id=tr-6a28e9b67536f991ab90530a7134d727)

結合上週的內容，將這個Pipeline打包成一個Artifact上傳到MLflow Server，然後藉由MLflow調用Pipeline

## Upload model as a python script

🎯 **本章學完你將能學會什麼：**

- 理解 MLflow ModelSignature 的用途與 schema 定義方法  
- 學會如何設定模型的輸入與輸出格式，確保部署時類型驗證正確  
- 掌握將模型與源代碼上傳為 Artifact 的流程  
- 瞭解如何註冊與載入 MLflow 模型以進行推論  

📘 **最終你將具備的能力：**  
能夠定義、封裝並上傳可重用的 LLM 模型，並以 MLflow 作為模型版本與部署的核心平台。


### ModelSignature

    ModelSignature 是 MLflow 的一個物件。在 MLflow 裡的主要作用，正是用來定義與驗證模型的輸入與輸出格式（schema）。 描述了模型預期的 輸入資料型態與欄位結構，以及模型會輸出的 結果型態與欄位結構

    在 MLflow 的 log_model 時標明 signature，主要是為了清楚定義模型的輸入與輸出格式。這對模型管理、部署以及後續使用都有實際好處

    如果不指定，MLflow 會嘗試自動推斷模型的輸入/輸出格式，但這有幾個問題：

    - 推斷可能不準確（特別是在複雜 pipeline 或輸入非標準 pandas/numpy 物件時）。

    - 模型被部署或交付給其他人時，使用者可能不知道要如何準備正確的輸入資料。

    因此 明確指定 signature 可以避免模糊不清。

    好處:

        - 可讀性與可解釋性

            清楚描述「這個模型吃什麼、吐什麼」，讓其他人（或未來的自己）一眼看懂。

        - 驗證輸入輸出

            MLflow 的 pyfunc API 會利用 signature 驗證輸入資料是否符合定義，避免「欄位缺失」或「型態不符」的錯誤。

        - 提升可移植性

            模型部署到 MLflow Serving 或其他 REST API 時，會自動帶上 schema 文件，API 使用者可以直接參考。

        - 幫助自動化工具

            在 AutoML pipeline 或 MLflow Model Registry 中，signature 可以被用來檢查模型相容性（例如 pipeline 不同階段之間輸入/輸出格式是否一致）。

    它讓 MLflow 能：
    
        - 在 模型儲存（log_model / save_model） 時記錄這些資訊。
    
        - 在 模型部署或推論（predict） 時自動檢查輸入資料是否符合定義。
    
        - 在 MLflow UI 中清楚顯示模型的「輸入/輸出結構」


這個例子中模型需要兩個輸入欄位：

- title（型別：string）

- article（型別：string）

模型會輸出一個欄位：

- 無名稱（或預設名稱）但型別是 string。

換句話說，這個 signature 明確說明了模型的 輸入結構 與 輸出結構，
可幫助 MLflow 在紀錄或部署模型時自動進行型別驗證與追蹤。

In [15]:
from textwrap import dedent

import pandas as pd
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

from src.io.path_definition import get_project_dir

model_path = os.path.join(get_project_dir(), 'tutorial', "week_4", "llmchain_mlflow_experiment_tracing.py")

# You need to know what you will put into it and what you will get out of it.
input_schema = Schema([ColSpec("string", "title"),
                       ColSpec("string", "article")])
output_schema = Schema([ColSpec("string")])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)


query = dedent("""
俗話說：「龍生九子，各有不同。」在廣闊浩瀚的海洋之中，就有一頭孤獨的鯨魚——五十二赫茲鯨魚。牠聲音的頻率天生便比同伴還要高，這項特別之處，也導致了牠與同伴產生了無法溝通的鴻溝。看見這則故事的我，不禁思考，在如此多元的人間，是否也有像五十二赫茲鯨魚一般，天生便與眾不同？

回首童年，我印象最為深刻的一刻，是初識字時，與文字互相理解的那一瞬、是當我第一次讀完一個句子時，它將自身的意義傳入我腦中的那一瞬。自此，我便對文字、語言抱有特殊的感情，也十分享受閱讀與朗誦。那種將自身與文字經由一點一滴積累而連接起來的感情，使我心靈感到十分富足。

而當我步入校園接觸同儕時，驀然驚覺我與別人的閱讀速度十分不同。每當我已讀完一篇文章，但同學可能只完成了一半甚至三分之二。同時，我在生字讀音方面也異常的執著，因此被同學抱怨有「文字潔癖」。面對同儕抱怨的我，也只好強忍對耳邊時而出現字錯讀音的不適，開始刻意忽略心裡對它的執念，只為想要與別人一樣，想要和朋友互相理解。

直到多年前，因緣際會之下認識了「五十二赫茲」這獨特的存在。牠的身影在我心中烙下一道深刻的痕跡。因為牠，我開始接受自己與他人的不同；也因為牠，我明白了，我對文字的執著，並不是一種負面的特質，而是上天賜予我的禮物，我開始在寫作上揮灑自如。這讓我知道，不要在一開始便用否定的眼光看待自己的特質。也許這特別之處，會使我們與五十二赫茲鯨魚一般孤獨，會使我們遭受他人的不理解與排斥，但也會讓我們與眾不同。

關於此，我想說的是，勇敢地綻放自己的特別，也讓自己成為自己和世人眼中，最閃耀的五十二赫茲鯨魚。
""")

run_name = "Reflection_Revision_Py"

with mlflow.start_run(run_name=run_name) as run:

    os.environ['experiment'] = experiment
    os.environ['run_id'] = run.info.run_id
    os.environ['run_name'] = run_name
    
    mlflow.log_artifact(model_path, artifact_path="source_code")

    input_example = pd.DataFrame(data=[[query, "關於五十二赫茲，我想說的是…"]], columns=['article', 'title'])
    
    model_info = mlflow.pyfunc.log_model(
        python_model=model_path,  # 將一個python檔案路徑當作"模型"
        name="langchain_model",
        input_example=input_example,
        signature=signature,
        registered_model_name="Generation_Reflection_Demo"
    )

2026/03/28 06:24:22 INFO mlflow.pyfunc: Validating input example against model signature
Could not import spacy python package. Please install it with `pip install spacy`.
Could not import textstat python package. Please install it with `pip install textstat`.
2026/03/28 06:25:13 WARNING mlflow.tracing.fluent: Failed to start span RunnableSequence: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.
2026/03/28 06:25:39 INFO mlflow.models.model: Found the following environment variables used during model inference: [OPENAI_API_KEY]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
Registered model 'Generation_Reflection_Demo' already exists. Creating a new version of this model...
2026/03/28 06:25:40 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name

🏃 View run Reflection_Revision_Py at: http://127.0.0.1:8080/#/experiments/1/runs/196c450d165e493cb608f2d2108c5d8e
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


Created version '2' of model 'Generation_Reflection_Demo'.


https://vocus.cc/article/67a5d9c0fd89780001f56271

In [16]:
query = dedent("""\
在茫茫大海中，有一隻鯨魚，看似平平無奇，卻是全世界最特別的鯨魚。牠就是五十二赫茲鯨魚，雖然一樣都會發出叫聲，但卻因頻率太高，所以沒有其他的鯨魚聽得見牠的聲音。因此，五十二赫茲鯨魚也代表自己獨特的存在。每個人心中，都有一頭這樣的鯨魚，而我也不例外。

從小，不知道是否因為社區有許多人飼養動物，我一直對動物都抱持著高度的熱情，只要在路上遇到了小動物，我便會欣喜若狂，迫不及待的上前觀察。每次都會詢問飼主關於動物的知識，每次都滿載而歸的我，心裡都會因更了解動物而幸福。

身為「動物學家」的我漸漸長大，不知不覺上了小學，雖然年歲增長了，但愛研究動物的精神依然還在。在日復一日的校園生活中，我每天都和同學報告「調查結果」。然而，久而久之，同學對我的行為感到厭煩，甚至有些人是懼怕動物的，大家認為我是否除了動物以外的知識全然不知，也漸漸疏遠了我，而我也開始懷疑自己是不是一個奇怪的人。

終於，我上了國中，接觸了新的科目，有一天，上生物課時，當老師正在介紹動物的身體構造，我赫然發現，我以前所研究的動物知識全都派上用場了。我欣喜若狂，並在那次的段考中名列前茅。我知道，我的獨特，絕對是幫助我的一對翅膀。

每個人心中都有一頭五十二赫茲鯨魚，牠代表了我們的獨特，我們的與眾不同。這些「獨特」是上天送給我們的禮物。因此，不要嫌棄並懷疑它的功用。我獨特的「動物學家」技能，讓我過著新奇的生活，使我天天都被知識包圍。對我來說，我的獨特，便是使我開心的種子，使我知識淵博的伙伴！

""")


with mlflow.start_run(run_name=run_name) as run:

    os.environ['experiment'] = experiment
    os.environ['run_id'] = run.info.run_id
    os.environ['run_name'] = run_name

    loaded_model = mlflow.pyfunc.load_model("models:/Generation_Reflection_Demo/2")
    
    input_ = pd.DataFrame(data=[[query, '關於五十二赫茲，我想說的是…']], columns=['article', 'title'])
    
    output = loaded_model.predict(input_)

Could not import spacy python package. Please install it with `pip install spacy`.
Could not import textstat python package. Please install it with `pip install textstat`.


🏃 View run Reflection_Revision_Py at: http://127.0.0.1:8080/#/experiments/1/runs/df50d3b026c74179be60ae12c17c0fba
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


Trace(trace_id=tr-87368cf564d549b5c74e2923966b4ba5)

In [17]:
print(output)

在茫茫大海中，有一隻鯨魚，看似平平無奇，卻是全世界最特別的鯨魚。牠就是五十二赫茲鯨魚，雖然一樣會發出叫聲，但因為頻率太高，其他鯨魚聽不見牠的聲音。因此，五十二赫茲鯨魚象徵著獨特的存在。每個人心中，都有一頭這樣的鯨魚，而我也不例外。

從小，我對動物抱持著高度的熱情，或許是因為社區裡有許多人飼養動物。每當在路上遇到小動物，我總是欣喜若狂，迫不及待地上前觀察。我會詢問飼主關於動物的知識，並且每次都滿載而歸，心裡因為更了解動物而感到幸福。

隨著年齡增長，我漸漸成為一名「動物學家」。在小學的日子裡，我每天都和同學報告我的「調查結果」。然而，久而久之，同學們對我的行為感到厭煩，甚至有些人對動物感到懼怕。他們開始質疑我是否除了動物以外的知識全然不知，漸漸疏遠了我。我也開始懷疑自己是不是一個奇怪的人。

終於，我上了國中，接觸了新的科目。有一天，在生物課上，當老師介紹動物的身體構造時，我驚喜地發現，過去研究的動物知識派上了用場。我在那次的段考中名列前茅，心中充滿了自豪。我知道，我的獨特，絕對是幫助我的一對翅膀。

每個人心中都有一頭五十二赫茲鯨魚，牠代表了我們的獨特與與眾不同。這些「獨特」是上天送給我們的禮物，因此，不要嫌棄並懷疑它的功用。我獨特的「動物學家」技能，讓我過著新奇的生活，讓我每天都被知識包圍。對我來說，我的獨特是使我開心的種子，是我知識淵博的伙伴！未來，我希望能夠將這份熱愛延續下去，探索更多動物的奧秘，並與他人分享這份喜悅。


## 多檔案的情況

當你的項目足夠大的時候，可能需要將代碼分散到多個檔案之中。

我們將 llmchain_mlflow_experiment_tracing.py 這個檔案稍微拆分一下

multifile_append_1.py 負責 build_standard_chat_prompt_template 這個函數
multifile_append_2.py 負責 Output 這個 結構化輸出格式

In [18]:
model_path = os.path.join(get_project_dir(), 'tutorial', "week_4", "multifile_core.py")

model_path_append_1 =  os.path.join(get_project_dir(), 'tutorial', "week_4", "multifile_append_1.py")
model_path_append_2 =  os.path.join(get_project_dir(), 'tutorial', "week_4", "multifile_append_2.py")

with mlflow.start_run(run_name='multi_experiment'):
    
    model_info = mlflow.pyfunc.log_model(
        python_model=model_path,  # 將一個python檔案路徑當作"模型"
        name="multifile_experiment",
        signature=signature,
        input_example=input_example,
        code_paths=[model_path_append_1, model_path_append_2],  # Include dependency
        registered_model_name="multifile_basic" 
    )

C:\Users\Ling\miniconda3\envs\aicg\lib\site-packages\mlflow\pyfunc\__init__.py:3304: UserWarning: An input example was not provided when logging the model. To ensure the model signature functions correctly, specify the `input_example` parameter. See https://mlflow.org/docs/latest/model/signatures.html#model-input-example for more details about the benefits of using input_example.
  color_warning(
Registered model 'multifile_basic' already exists. Creating a new version of this model...
2026/03/28 06:37:37 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: multifile_basic, version 4
Created version '4' of model 'multifile_basic'.


🏃 View run multi_experiment at: http://127.0.0.1:8080/#/experiments/1/runs/0a2f0a0006404d97b72bc43eb4d96daf
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


In [19]:
model_loaded = mlflow.pyfunc.load_model(model_uri="models:/multifile_basic/4")

Could not import spacy python package. Please install it with `pip install spacy`.
Could not import textstat python package. Please install it with `pip install textstat`.


In [20]:
input_ = pd.DataFrame(data=[[query, '關於五十二赫茲，我想說的是…']], columns=['article', 'title'])
    
output = loaded_model.predict(input_)

Trace(trace_id=tr-01b6e3de07dcb76aaec5cef936988cd3)

試著讓 

model_path_append_1 =  os.path.join(get_project_dir(), 'tutorial', "week_4", "multifile_append_1.py")
model_path_append_2 =  os.path.join(get_project_dir(), 'tutorial', "week_4", "multifile_append_2.py")

這兩個檔案消失在視野，看看是否還能正常執行

In [21]:
model_loaded = mlflow.pyfunc.load_model(model_uri="models:/multifile_basic/4")

input_ = pd.DataFrame(data=[[query, '關於五十二赫茲，我想說的是…']], columns=['article', 'title'])
    
output = loaded_model.predict(input_)

Could not import spacy python package. Please install it with `pip install spacy`.
Could not import textstat python package. Please install it with `pip install textstat`.


Trace(trace_id=tr-08a902d7ceea65d943164172ee0b32a3)

In [22]:
output

'在茫茫大海中，有一隻鯨魚，看似平平無奇，卻是全世界最特別的鯨魚。牠就是五十二赫茲鯨魚，雖然一樣會發出叫聲，但因為頻率太高，其他鯨魚聽不見牠的聲音。因此，五十二赫茲鯨魚象徵著獨特的存在。每個人心中，都有一頭這樣的鯨魚，而我也不例外。\n\n從小，我對動物抱持著高度的熱情，或許是因為社區裡有許多人飼養動物。每當在路上遇到小動物，我總是欣喜若狂，迫不及待地上前觀察。我會詢問飼主關於動物的知識，並且每次都滿載而歸，心裡因為更了解動物而感到幸福。\n\n隨著年齡增長，我漸漸成為一名「動物學家」。在小學的日子裡，我每天都和同學報告我的「調查結果」。然而，久而久之，同學們對我的行為感到厭煩，甚至有些人對動物感到懼怕。他們開始質疑我是否除了動物以外的知識全然不知，漸漸疏遠了我。我也開始懷疑自己是不是一個奇怪的人。\n\n終於，我上了國中，接觸了新的科目。有一天，在生物課上，當老師介紹動物的身體構造時，我驚喜地發現，過去研究的動物知識派上了用場。我在那次的段考中名列前茅，心中充滿了自豪。我知道，我的獨特，絕對是幫助我的一對翅膀。\n\n每個人心中都有一頭五十二赫茲鯨魚，牠代表了我們的獨特與與眾不同。這些「獨特」是上天送給我們的禮物，因此，不要嫌棄並懷疑它的功用。我獨特的「動物學家」技能，讓我過著新奇的生活，讓我每天都被知識包圍。對我來說，我的獨特，便是使我開心的種子，使我知識淵博的伙伴！未來，我希望能夠將這份熱情延續下去，探索更多動物的奧秘，並與他人分享這份喜悅。'

換句話說，這些"code_paths中的附屬檔案已經被一併打包成為了模型的一部分"

## MLflow 數據追蹤

In [23]:
# 使用一份數據
filename = os.path.join(get_project_dir(), 'tutorial', "week_4", "benchmark_evaluation.csv")
raw_data = pd.read_csv(filename)

In [24]:
# 建立一個dataset 物件
dataset = mlflow.data.from_pandas(
    raw_data, source=filename, name="benchmark"
)

C:\Users\Ling\miniconda3\envs\aicg\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


In [25]:
with mlflow.start_run(run_name="data tracing"):
    mlflow.log_input(dataset, context="training")

🏃 View run data tracing at: http://127.0.0.1:8080/#/experiments/1/runs/0d9435716c1842d88ac160d5e1e64273
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


但是這個方法只會追蹤數據的Metadata

- Dataset name
- Dataset source
- Dataset schema (columns, types)
- Dataset digest / fingerprint
- Dataset context (training / validation / etc.)

⚠️ 常見誤解澄清：

- `mlflow.log_input()` **不會自動保存原始資料內容**
- 它的目的在於：
  - 建立「模型 ↔ 資料」的血緣關係
  - 確保實驗可追蹤、可稽核

👉 若你需要真正保存資料：
- 必須額外使用 `mlflow.log_artifact()`
- 或搭配外部資料儲存（S3 / GCS / DB）

這也是為什麼實務上常見做法是：
- Dataset lineage（metadata）→ MLflow
- Dataset 本體 → Artifact 或 Data Platform

## MLflow 數據上傳

In [26]:
uploaded_filename = os.path.join(get_project_dir(), "tutorial", "week_4", "benchmark.parquet")

with mlflow.start_run(run_name='data_upload'):
    mlflow.log_input(dataset, context="training_2")
    raw_data.to_parquet(uploaded_filename) 
    mlflow.log_artifact(uploaded_filename, artifact_path="data")

🏃 View run data_upload at: http://127.0.0.1:8080/#/experiments/1/runs/f2d21e85e7c44ca382e6b287d85d1b7f
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


接著我們就可以根據Run ID 下載 這個檔案

將指定的檔案下載到downloads這個資料夾中

In [27]:
mlflow.artifacts.download_artifacts(run_id="f2d21e85e7c44ca382e6b287d85d1b7f",
                                   artifact_path="data/benchmark.parquet",
                                   # 相對於工作資料夾的位置
                                   dst_path="tutorial/week_4/downloads")

'C:\\Users\\Ling\\Dev\\RookieSavior_LLM_Applications\\tutorial\\week_4\\downloads\\benchmark.parquet'

我們可以看到，要下載artifacts需要一組run_id。
若是我們不想要MLflow的UI使用人肉搜索找出數據的run_id的話，有以下幾種手段

### 模型綁定

模型通常會跟數據綁定(像是在做RAG的時候)。所以登記模型的時候也同時登記數據

這只是一個示範，不代表這個模型和數據綁在一起。

In [29]:
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

input_schema = Schema([ColSpec("string")])
output_schema = Schema([ColSpec("string")])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

filename = os.path.join(get_project_dir(), 'tutorial', "week_4", "benchmark_evaluation.csv")

with mlflow.start_run(run_name='method_1') as run:
    # 設定一組 run tag. 等等會用到
    mlflow.set_tag("Hello", "World")
    
    # 1. Log dataset lineage
    dataset = mlflow.data.from_pandas(
        raw_data,
        source=filename,
        name="benchmark"
    )

    # 紀錄 Metadata
    mlflow.log_input(dataset, context="training")

    # 2. Log dataset artifact
    raw_data.to_parquet(uploaded_filename)
    mlflow.log_artifact(uploaded_filename, artifact_path="data")
    
    model_info = mlflow.pyfunc.log_model(
        python_model=model_path,
        name="multifile experiment",
        signature=signature,
        input_example=input_example,
        code_paths=[model_path_append_1, model_path_append_2],  # Include dependency
        registered_model_name="multifile_basic" 
    )

C:\Users\Ling\miniconda3\envs\aicg\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
2026/03/28 07:08:45 INFO mlflow.pyfunc: Validating input example against model signature
Could not import spacy python package. Please install it with `pip install spacy`.
Could not import textstat python package. Please install it with `pip install textstat`.
2026/03/28 07:09:27 WARNING mlflow.models.model: Failed to validate serving input example {
  "dataframe_split": {
    "columns": [
      "a.... Alternatively, you can avoid passing input example and pass model signature instead when logging the model. To ensure the input example is valid prior to serving, please try calling `mlflow.models.validate_serving_input` on the model uri and serving 

🏃 View run method_1 at: http://127.0.0.1:8080/#/experiments/1/runs/7617268f2fff4f3ea986e48702882984
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


Created version '5' of model 'multifile_basic'.


這樣子我們就可以透過模型版本追蹤數據

In [30]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

mv = client.get_model_version(
    name="multifile_basic",
    version="5")

mv.run_id

'7617268f2fff4f3ea986e48702882984'

### 透過實驗和數據的名稱取得數據
1. **取得實驗id**

In [31]:
client = MlflowClient()

experiment = "week_4"
tracking_url = "http://127.0.0.1:8080"

experiment = client.get_experiment_by_name(experiment)  # or your experiment name
experiment_id = experiment.experiment_id

print(experiment_id)

1


2 **搜尋最新的artifacts**

In [32]:
runs = client.search_runs(
    experiment_ids=[experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["attributes.start_time DESC"],
    max_results=10
)

3. **選擇你要的run**

In [33]:
runs[0].info.run_id

'7617268f2fff4f3ea986e48702882984'

**透過數據的metadata**

In [34]:
runs = client.search_runs(
    experiment_ids=[experiment_id],
    filter_string="dataset.name = 'benchmark'"
)

runs[0].info.run_id

'7617268f2fff4f3ea986e48702882984'

**透過run的tag**

In [ ]:
# 先選好模型，理想中是在上傳模型時就設定好了

# client.set_model_version_tag(
#     name="multifile_basic",
#     version="5",
#     key="Hello",
#     value="World"
# )

In [35]:
runs = client.search_runs(
    experiment_ids=[experiment_id],
    filter_string="tags.Hello = 'World'"
)

In [36]:
runs[0].info.run_id

'7617268f2fff4f3ea986e48702882984'

所以做好登錄紀錄可以幫助在之後快速找到相關的數據

# LangServe API

🎯 **本章學完你將能學會什麼：**

- 理解 LangServe 的架構與 API 調用流程  
- 學會建立簡單的後端伺服器，並從客戶端發送請求取得模型回應  
- 掌握 RemoteRunnable 的應用，讓模型能夠遠端呼叫  
- 瞭解如何結合 MLflow 模型與 LangServe API 進行整合部署  

📘 **最終你將具備的能力：**  
能夠設計一個具備後端 API 介面的 LLM 系統，支援遠端推論與模組化部署。  

## 1. 客戶端 (client) 呼叫後端 API

1. 確認開啟MLFlow
2. 開啟檔案 app_basics.py

In [37]:
import requests

response = requests.post(
    "http://localhost:5000/openai/invoke",
    json={'input': "Where is Taiwan?"}
)

In [38]:
response.json()

{'output': {'content': 'Taiwan is an island located in East Asia, situated in the western Pacific Ocean. It lies off the southeastern coast of China, separated by the Taiwan Strait. To the north of Taiwan is the East China Sea, and to the south is the Bashi Channel, which separates it from the Philippines. Taiwan is known for its mountainous terrain, vibrant cities, and rich cultural heritage. The capital city is Taipei, located in the northern part of the island.',
  'additional_kwargs': {'refusal': None},
  'response_metadata': {'token_usage': {'completion_tokens': 91,
    'prompt_tokens': 11,
    'total_tokens': 102,
    'completion_tokens_details': {'accepted_prediction_tokens': 0,
     'audio_tokens': 0,
     'reasoning_tokens': 0,
     'rejected_prediction_tokens': 0},
    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}},
   'model_provider': 'openai',
   'model_name': 'gpt-4o-mini-2024-07-18',
   'system_fingerprint': 'fp_c93645df76',
   'id': 'chatcmpl-DOHMgrrB

In [39]:
response.json()['output']['content']

'Taiwan is an island located in East Asia, situated in the western Pacific Ocean. It lies off the southeastern coast of China, separated by the Taiwan Strait. To the north of Taiwan is the East China Sea, and to the south is the Bashi Channel, which separates it from the Philippines. Taiwan is known for its mountainous terrain, vibrant cities, and rich cultural heritage. The capital city is Taipei, located in the northern part of the island.'

In [40]:
response = requests.post(
    "http://localhost:5000/openai/invoke",
    json={'input': "花蓮縣光復鄉因為馬太鞍溪堰塞湖潰堤，導致被泥石流淹過。就安全的考量，沒接受過專業訓練的平民是否應該去花蓮縣光復鄉參與救災?"}
)

response.json()['output']['content']

'在面對自然災害和救災行動時，安全是最重要的考量。對於沒有接受過專業訓練的平民，參與救災可能會帶來風險，因為他們可能無法有效應對潛在的危險情況，如泥石流、倒塌的建築物或其他危險環境。\n\n如果您沒有相關的專業知識和經驗，建議您不要直接參與救災行動。相反，您可以考慮以下幾種方式來提供支持：\n\n1. **捐款或捐物**：向專業的救災機構或慈善組織捐款或捐贈物資，這樣可以更有效地幫助受災者。\n\n2. **志願服務**：如果您有相關的技能或經驗，可以尋找當地的志願者組織，看看是否有適合您的角色。\n\n3. **提供心理支持**：在災後，心理支持也非常重要，您可以考慮提供情感支持或陪伴。\n\n4. **遵循官方指示**：密切關注當地政府和救災機構的公告，遵循他們的指示和建議。\n\n總之，雖然想要幫助是非常可貴的，但在沒有專業訓練的情況下，最安全的做法是尋找其他方式來支持救災工作。'

在Windows的CLI(command line interface)中:

curl -X POST "http://localhost:5000/openai/invoke" -H "Content-Type: application/json" -d "{""input"": ""Where is Taiwan?""}"

## 2. 結合之前MLflow的應用。從MLflow server上下載模型，然後從客戶端呼叫

input_ = pd.DataFrame(data=[[query, '關於五十二赫茲，我想說的是…']], columns=['article', 'title'])
    
output = loaded_model.predict(input_)

In [41]:
response = requests.post(
    "http://localhost:5000/demo/invoke",
    json={'input': {"article": query,
                    "title": '關於五十二赫茲，我想說的是…'}}
)

In [42]:
response.json()

{'output': '在茫茫大海中，有一隻鯨魚，看似平平無奇，卻是全世界最特別的鯨魚。牠就是五十二赫茲鯨魚，雖然一樣會發出叫聲，但因為頻率太高，其他鯨魚聽不見牠的聲音。因此，五十二赫茲鯨魚象徵著獨特的存在。每個人心中，都有一頭這樣的鯨魚，而我也不例外。\n\n從小，我對動物抱持著高度的熱情，或許是因為社區裡有許多人飼養動物。每當在路上遇到小動物，我總是欣喜若狂，迫不及待地上前觀察。我會詢問飼主關於動物的知識，並且每次都滿載而歸，心裡因為更了解動物而感到幸福。\n\n隨著年齡增長，我漸漸成為一名「動物學家」。在小學的日子裡，我每天都和同學報告我的「調查結果」。然而，久而久之，同學們對我的行為感到厭煩，甚至有些人對動物感到懼怕。他們開始質疑我是否除了動物以外的知識全然不知，漸漸疏遠了我。我也開始懷疑自己是不是一個奇怪的人。\n\n終於，我上了國中，接觸了新的科目。有一天，在生物課上，老師介紹動物的身體構造時，我赫然發現我以前所研究的動物知識全都派上用場了。我欣喜若狂，並在那次的段考中名列前茅。我知道，我的獨特，絕對是幫助我的一對翅膀。\n\n每個人心中都有一頭五十二赫茲鯨魚，牠代表了我們的獨特與與眾不同。這些「獨特」是上天送給我們的禮物，因此，不要嫌棄並懷疑它的功用。我獨特的「動物學家」技能，讓我過著新奇的生活，使我天天都被知識包圍。對我來說，我的獨特，便是使我開心的種子，也是我知識淵博的伙伴！',
 'metadata': {'run_id': '40f87070-b4a9-42ea-9b49-376117845c11',
  'feedback_tokens': []}}

## 3 RemoteRunnable

In [43]:
from langserve import RemoteRunnable

remote_llm = RemoteRunnable("http://localhost:5000/openai/")

In [44]:
# Supports astream
async for msg in remote_llm.astream("Where is Taiwan?"):
    print(msg.content, end="", flush=True)

Taiwan is an island located in East Asia, situated in the western Pacific Ocean. It lies off the southeastern coast of China, separated by the Taiwan Strait. To the north of Taiwan is the East China Sea, while to the south is the Bashi Channel, which separates it from the Philippines. Taiwan is known for its mountainous terrain, vibrant cities, and rich cultural heritage. The capital city of Taiwan is Taipei.

2026/03/28 07:34:40 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x0000029EF32A56C0> at 0x0000029EC4152F80> was created in a different Context


Trace(trace_id=tr-d2f922d0f3f9f9f3cd891f49cd236140)

# ChatBot 本體與記憶機制

🎯 **本章學完你將能學會什麼：**

- 了解 Stateless 與 Stateful 對話系統的差異    
- 掌握 ChatPromptTemplate 與 MessagesPlaceholder 的應用  
- 能實作簡單的多輪對話機制 

📘 **最終你將具備的能力：**  
能夠設計一個具備「對話記憶」的智能 ChatBot，能夠在上下文中延續並理解對話語境。  


1. N-Shot Learning 與對話歷史

    - 歷史對話可以看成一個 Q&A pair 列表
    - 當前模型在推理時，會把「之前的對話內容」作為 prompt 的一部分，再加上使用者最新的輸入，整合後丟進模型。這其實就是一種 few-shot / N-shot 的學習方式：模型從範例中抽取語境來理解「現在該怎麼回答」。

2. Stateless vs. Stateful
    - Stateless：如果每次請求都完全獨立，沒有任何歷史對話被帶入，那就叫無狀態 (stateless)。
    - Stateful：如果系統會保存對話歷史（不論是把歷史傳回模型，還是外部記憶系統存起來），那就是有狀態的 (stateful)。
    - 所以是否「能記住」過去，取決於設計，而不是模型本身自帶的能力。

3. Tools 的角色
    - 讓 ChatBot 強大的是 tools
    - 模型本身雖然能生成語言，但 結合外部工具（例如資料庫查詢、計算器、網路搜尋、代碼執行、圖片生成）後，ChatBot 才能真正做到「會推理 + 會行動」，不再只受限於參數內的知識。

    - 可以理解成：模型是「大腦」，Tools 是「手腳」。

In [ ]:
from IPython.display import Image

Image(url="https://python.langchain.com/v0.1/assets/images/chat_use_case-eb8a4883931d726e9f23628a0d22e315.png")

先學怎麼調動工具: 模型就像是一個訓練有素的阿斯塔特，工具就像是動力甲，噴射背包，爆彈槍，和鏈鋸劍。

## 工具綁定與工具呼叫 (Tools & ToolMessage)

🎯 **本章學完你將能學會什麼：**

- 理解 Tool 與 LLM 的互動方式  
- 學會使用 LangChain 的 @tool 與 StructuredTool 綁定外部函式  
- 掌握 ToolMessage 的設計與呼叫邏輯  
- 能整合多個工具（如計算、查詢、WebSearch）讓模型具備「行動能力」  

📘 **最終你將具備的能力：**  
能夠打造能「思考 + 行動」的 ChatBot，結合多種工具完成自動化任務。  

In [45]:
import os

# os.chdir("../../")

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

from initialization import credential_init

credential_init()

# Define a calculator tool
@tool
def add_numbers(a: int, b: int) -> int:
    """Adds two numbers together."""
    return a + b

# # Create the LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# Bind the tool to the model
llm_with_tools = llm.bind_tools([add_numbers])

# Run
resp = llm_with_tools.invoke("What is 42 + 58?")

Trace(trace_id=tr-f7a4d333a319038239792208c20fed58)

In [46]:
resp

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 54, 'total_tokens': 72, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_cdd011970c', 'id': 'chatcmpl-DOHfx19ZawlAd1HyxZmN7vThlII0p', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--e6088280-089f-429d-9fb6-c3e7470da2af-0', tool_calls=[{'name': 'add_numbers', 'args': {'a': 42, 'b': 58}, 'id': 'call_c68KJhkCzC4Ok6zDfoQNko9K', 'type': 'tool_call'}], usage_metadata={'input_tokens': 54, 'output_tokens': 18, 'total_tokens': 72, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [47]:
tool_call = resp.tool_calls[0]
print(tool_call)

{'name': 'add_numbers', 'args': {'a': 42, 'b': 58}, 'id': 'call_c68KJhkCzC4Ok6zDfoQNko9K', 'type': 'tool_call'}


根據tool_call計算結果

In [48]:
result = eval(tool_call['name']).invoke(tool_call['args'])

Trace(trace_id=tr-71515eaf6f4f3d9e860a672ff5daf6b8)

In [49]:
result

100

建立ToolMessage

In [50]:
from langchain_core.messages import ToolMessage

tool_msg = ToolMessage(
    content=str(result),          # usually a string or simple text
    tool_call_id=tool_call['id']    # must match the AIMessage tool_call id
)

In [51]:
resp

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 54, 'total_tokens': 72, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_cdd011970c', 'id': 'chatcmpl-DOHfx19ZawlAd1HyxZmN7vThlII0p', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--e6088280-089f-429d-9fb6-c3e7470da2af-0', tool_calls=[{'name': 'add_numbers', 'args': {'a': 42, 'b': 58}, 'id': 'call_c68KJhkCzC4Ok6zDfoQNko9K', 'type': 'tool_call'}], usage_metadata={'input_tokens': 54, 'output_tokens': 18, 'total_tokens': 72, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [52]:
tool_msg

ToolMessage(content='100', tool_call_id='call_c68KJhkCzC4Ok6zDfoQNko9K')

最後，綁定AIMessage和ToolMessage，在進行一次invoke得到結果

In [53]:
llm_with_tools.invoke([resp, tool_msg])

AIMessage(content='The sum of 42 and 58 is 100.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 69, 'total_tokens': 82, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_cdd011970c', 'id': 'chatcmpl-DOHkvtrZvMve2kExx8grBcV3fpUOp', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--65436df0-a066-4b05-a0cb-79eeaef13cdd-0', usage_metadata={'input_tokens': 69, 'output_tokens': 13, 'total_tokens': 82, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Trace(trace_id=tr-113779bb049ae0634380e9323e898256)

## OpenAI WebSearch API 與網路搜尋應用

🎯 **本章學完你將能學會什麼：**

- 學會調用 OpenAI WebSearch 工具取得即時資料  
- 理解 WebSearch API 的 action、annotations、sources 參數  
- 掌握如何在回應中引用來源並分析結果  
- 了解 WebSearch 的優缺點與應用策略  

📘 **最終你將具備的能力：**  
能夠設計可即時檢索、分析並生成報告的 AI 助手，並根據情境選擇最佳資訊來源。  


### 基本使用

In [55]:
from openai import OpenAI

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

In [56]:
response = client.responses.create(
         model="gpt-4o-mini",
         tools=[{"type": "web_search"}],
         input="幫我查詢HunterXHunter 最新的進度"
    )

response

Response(id='resp_03cd89106b7498830069c77a3f089c819fbbb870602254949e', created_at=1774680639.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseFunctionWebSearch(id='ws_03cd89106b7498830069c77a3fe3a0819faa1e99d45442af1f', action=ActionSearch(query='Hunter x Hunter 最新進度 2023', type='search', queries=['Hunter x Hunter 最新進度 2023'], sources=None), status='completed', type='web_search_call'), ResponseOutputMessage(id='msg_03cd89106b7498830069c77a40dd80819fabb2165428245db5', content=[ResponseOutputText(annotations=[AnnotationURLCitation(end_index=253, start_index=182, title='《獵人》漫畫進度飆車！冨樫29天狂畫4話 庫拉皮卡現身粉絲嗨翻 | 生活 | NOWnews今日新聞', type='url_citation', url='https://www.nownews.com/news/6775626?utm_source=openai'), AnnotationURLCitation(end_index=425, start_index=316, title='睽違1年10個月，《獵人》新單行本9/4發售 | 4Gamers', type='url_citation', url='https://www.4gamers.com.tw/news/detail/66089/hunter-x--hunter-new-comic?utm_s

In [57]:
response.output

[ResponseFunctionWebSearch(id='ws_03cd89106b7498830069c77a3fe3a0819faa1e99d45442af1f', action=ActionSearch(query='Hunter x Hunter 最新進度 2023', type='search', queries=['Hunter x Hunter 最新進度 2023'], sources=None), status='completed', type='web_search_call'),
 ResponseOutputMessage(id='msg_03cd89106b7498830069c77a40dd80819fabb2165428245db5', content=[ResponseOutputText(annotations=[AnnotationURLCitation(end_index=253, start_index=182, title='《獵人》漫畫進度飆車！冨樫29天狂畫4話 庫拉皮卡現身粉絲嗨翻 | 生活 | NOWnews今日新聞', type='url_citation', url='https://www.nownews.com/news/6775626?utm_source=openai'), AnnotationURLCitation(end_index=425, start_index=316, title='睽違1年10個月，《獵人》新單行本9/4發售 | 4Gamers', type='url_citation', url='https://www.4gamers.com.tw/news/detail/66089/hunter-x--hunter-new-comic?utm_source=openai')], text='《獵人x獵人》（Hunter × Hunter）自1998年連載以來，因作者冨樫義博的健康狀況，連載進度時有中斷。然而，近期有關連載進度的更新值得關注。\n\n2025年12月15日，冨樫義博在X（前稱Twitter）上宣布第411話原稿已完成。隨後，在2026年1月14日，他再度更新，表示第414話原稿也已完成，並附上庫拉皮卡、西索等角色的插圖，令粉絲興奮不已。 ([nownews.com](

In [58]:
response.output[1]

ResponseOutputMessage(id='msg_03cd89106b7498830069c77a40dd80819fabb2165428245db5', content=[ResponseOutputText(annotations=[AnnotationURLCitation(end_index=253, start_index=182, title='《獵人》漫畫進度飆車！冨樫29天狂畫4話 庫拉皮卡現身粉絲嗨翻 | 生活 | NOWnews今日新聞', type='url_citation', url='https://www.nownews.com/news/6775626?utm_source=openai'), AnnotationURLCitation(end_index=425, start_index=316, title='睽違1年10個月，《獵人》新單行本9/4發售 | 4Gamers', type='url_citation', url='https://www.4gamers.com.tw/news/detail/66089/hunter-x--hunter-new-comic?utm_source=openai')], text='《獵人x獵人》（Hunter × Hunter）自1998年連載以來，因作者冨樫義博的健康狀況，連載進度時有中斷。然而，近期有關連載進度的更新值得關注。\n\n2025年12月15日，冨樫義博在X（前稱Twitter）上宣布第411話原稿已完成。隨後，在2026年1月14日，他再度更新，表示第414話原稿也已完成，並附上庫拉皮卡、西索等角色的插圖，令粉絲興奮不已。 ([nownews.com](https://www.nownews.com/news/6775626?utm_source=openai))\n\n此外，2024年7月22日，集英社官方網站公布，第38卷單行本將於2024年9月4日發售，距離上一卷發售已有近兩年之久。 ([4gamers.com.tw](https://www.4gamers.com.tw/news/detail/66089/hunter-x--hunter-new-comic?utm_source=openai))\n\n綜合以上資訊，冨樫義博近期在連載進度上有所突破

In [59]:
response.output[1].content

[ResponseOutputText(annotations=[AnnotationURLCitation(end_index=253, start_index=182, title='《獵人》漫畫進度飆車！冨樫29天狂畫4話 庫拉皮卡現身粉絲嗨翻 | 生活 | NOWnews今日新聞', type='url_citation', url='https://www.nownews.com/news/6775626?utm_source=openai'), AnnotationURLCitation(end_index=425, start_index=316, title='睽違1年10個月，《獵人》新單行本9/4發售 | 4Gamers', type='url_citation', url='https://www.4gamers.com.tw/news/detail/66089/hunter-x--hunter-new-comic?utm_source=openai')], text='《獵人x獵人》（Hunter × Hunter）自1998年連載以來，因作者冨樫義博的健康狀況，連載進度時有中斷。然而，近期有關連載進度的更新值得關注。\n\n2025年12月15日，冨樫義博在X（前稱Twitter）上宣布第411話原稿已完成。隨後，在2026年1月14日，他再度更新，表示第414話原稿也已完成，並附上庫拉皮卡、西索等角色的插圖，令粉絲興奮不已。 ([nownews.com](https://www.nownews.com/news/6775626?utm_source=openai))\n\n此外，2024年7月22日，集英社官方網站公布，第38卷單行本將於2024年9月4日發售，距離上一卷發售已有近兩年之久。 ([4gamers.com.tw](https://www.4gamers.com.tw/news/detail/66089/hunter-x--hunter-new-comic?utm_source=openai))\n\n綜合以上資訊，冨樫義博近期在連載進度上有所突破，粉絲們對未來的連載更新充滿期待。 ', type='output_text', logprobs=[])]

In [60]:
response.output[1].content[0]

ResponseOutputText(annotations=[AnnotationURLCitation(end_index=253, start_index=182, title='《獵人》漫畫進度飆車！冨樫29天狂畫4話 庫拉皮卡現身粉絲嗨翻 | 生活 | NOWnews今日新聞', type='url_citation', url='https://www.nownews.com/news/6775626?utm_source=openai'), AnnotationURLCitation(end_index=425, start_index=316, title='睽違1年10個月，《獵人》新單行本9/4發售 | 4Gamers', type='url_citation', url='https://www.4gamers.com.tw/news/detail/66089/hunter-x--hunter-new-comic?utm_source=openai')], text='《獵人x獵人》（Hunter × Hunter）自1998年連載以來，因作者冨樫義博的健康狀況，連載進度時有中斷。然而，近期有關連載進度的更新值得關注。\n\n2025年12月15日，冨樫義博在X（前稱Twitter）上宣布第411話原稿已完成。隨後，在2026年1月14日，他再度更新，表示第414話原稿也已完成，並附上庫拉皮卡、西索等角色的插圖，令粉絲興奮不已。 ([nownews.com](https://www.nownews.com/news/6775626?utm_source=openai))\n\n此外，2024年7月22日，集英社官方網站公布，第38卷單行本將於2024年9月4日發售，距離上一卷發售已有近兩年之久。 ([4gamers.com.tw](https://www.4gamers.com.tw/news/detail/66089/hunter-x--hunter-new-comic?utm_source=openai))\n\n綜合以上資訊，冨樫義博近期在連載進度上有所突破，粉絲們對未來的連載更新充滿期待。 ', type='output_text', logprobs=[])

In [61]:
response.output[1].content[0].text

'《獵人x獵人》（Hunter × Hunter）自1998年連載以來，因作者冨樫義博的健康狀況，連載進度時有中斷。然而，近期有關連載進度的更新值得關注。\n\n2025年12月15日，冨樫義博在X（前稱Twitter）上宣布第411話原稿已完成。隨後，在2026年1月14日，他再度更新，表示第414話原稿也已完成，並附上庫拉皮卡、西索等角色的插圖，令粉絲興奮不已。 ([nownews.com](https://www.nownews.com/news/6775626?utm_source=openai))\n\n此外，2024年7月22日，集英社官方網站公布，第38卷單行本將於2024年9月4日發售，距離上一卷發售已有近兩年之久。 ([4gamers.com.tw](https://www.4gamers.com.tw/news/detail/66089/hunter-x--hunter-new-comic?utm_source=openai))\n\n綜合以上資訊，冨樫義博近期在連載進度上有所突破，粉絲們對未來的連載更新充滿期待。 '

In [62]:
response.output[1].content[0].annotations

[AnnotationURLCitation(end_index=253, start_index=182, title='《獵人》漫畫進度飆車！冨樫29天狂畫4話 庫拉皮卡現身粉絲嗨翻 | 生活 | NOWnews今日新聞', type='url_citation', url='https://www.nownews.com/news/6775626?utm_source=openai'),
 AnnotationURLCitation(end_index=425, start_index=316, title='睽違1年10個月，《獵人》新單行本9/4發售 | 4Gamers', type='url_citation', url='https://www.4gamers.com.tw/news/detail/66089/hunter-x--hunter-new-comic?utm_source=openai')]

### Web 搜尋呼叫（`web_search_call`）輸出項目

一個 `web_search_call` 的輸出項目，包含搜尋呼叫的 ID，以及在 `web_search_call.action` 中所採取的動作。  
此動作可能為以下其中之一：

- **`search`**：  
  代表一次網頁搜尋。通常（但不一定）會包含搜尋查詢字串以及被搜尋的網域。  
  搜尋動作會產生工具呼叫成本（請參閱定價說明）。

- **`open_page`**：  
  代表開啟某個頁面。  
  在推理模型（reasoning models）中支援。

- **`find_in_page`**：  
  代表在頁面內進行搜尋。  
  在推理模型（reasoning models）中支援。

---

### 訊息（`message`）輸出項目

包含以下內容：

- **文字結果**：位於 `message.content[1].text`
- **註解**：位於 `message.content[0].annotations`，用於標註被引用的 URL


In [63]:
response.output[0]

ResponseFunctionWebSearch(id='ws_03cd89106b7498830069c77a3fe3a0819faa1e99d45442af1f', action=ActionSearch(query='Hunter x Hunter 最新進度 2023', type='search', queries=['Hunter x Hunter 最新進度 2023'], sources=None), status='completed', type='web_search_call')

In [64]:
response.output[1].content[0].annotations

[AnnotationURLCitation(end_index=253, start_index=182, title='《獵人》漫畫進度飆車！冨樫29天狂畫4話 庫拉皮卡現身粉絲嗨翻 | 生活 | NOWnews今日新聞', type='url_citation', url='https://www.nownews.com/news/6775626?utm_source=openai'),
 AnnotationURLCitation(end_index=425, start_index=316, title='睽違1年10個月，《獵人》新單行本9/4發售 | 4Gamers', type='url_citation', url='https://www.4gamers.com.tw/news/detail/66089/hunter-x--hunter-new-comic?utm_source=openai')]

websearch 結果:

In [65]:
response.output_text

'《獵人x獵人》（Hunter × Hunter）自1998年連載以來，因作者冨樫義博的健康狀況，連載進度時有中斷。然而，近期有關連載進度的更新值得關注。\n\n2025年12月15日，冨樫義博在X（前稱Twitter）上宣布第411話原稿已完成。隨後，在2026年1月14日，他再度更新，表示第414話原稿也已完成，並附上庫拉皮卡、西索等角色的插圖，令粉絲興奮不已。 ([nownews.com](https://www.nownews.com/news/6775626?utm_source=openai))\n\n此外，2024年7月22日，集英社官方網站公布，第38卷單行本將於2024年9月4日發售，距離上一卷發售已有近兩年之久。 ([4gamers.com.tw](https://www.4gamers.com.tw/news/detail/66089/hunter-x--hunter-new-comic?utm_source=openai))\n\n綜合以上資訊，冨樫義博近期在連載進度上有所突破，粉絲們對未來的連載更新充滿期待。 '

#### Source 參數
若要查看在網路搜尋過程中擷取的所有網址，可使用 sources 欄位。
與只顯示最相關參考資料的「行內引用（inline citations）」不同，sources 會回傳模型在生成回應時所參考的完整網址清單。

位於: response.output[0].action.sources

In [66]:
response = client.responses.create(
         model="gpt-4o-mini",
         tools=[{"type": "web_search"}],
         include=["web_search_call.action.sources"],
         input="幫我查詢HunterXHunter 最新的進度"
    )

# response

In [67]:
response.output[1].content[0].annotations

[AnnotationURLCitation(end_index=204, start_index=133, title='《獵人》漫畫進度飆車！冨樫29天狂畫4話 庫拉皮卡現身粉絲嗨翻 | 生活 | NOWnews今日新聞', type='url_citation', url='https://www.nownews.com/news/6775626?utm_source=openai'),
 AnnotationURLCitation(end_index=346, start_index=267, title='『HUNTER×HUNTER』この先50話分の進捗状況\u3000登場キャラ一部公開！冨樫義博「台詞と時系列の確認・調整中」 | オリコンニュース（ORICON NEWS）', type='url_citation', url='https://www.oricon.co.jp/news/2359490/full/?utm_source=openai'),
 AnnotationURLCitation(end_index=520, start_index=411, title='睽違1年10個月，《獵人》新單行本9/4發售 | 4Gamers', type='url_citation', url='https://www.4gamers.com.tw/news/detail/66089/hunter-x--hunter-new-comic?utm_source=openai')]

In [68]:
response.output[0].action.sources

[ActionSearchSource(type='url', url='https://www.oricon.co.jp/news/2359490/full/?utm_source=openai'),
 ActionSearchSource(type='url', url='https://www.nownews.com/news/6775626?utm_source=openai'),
 ActionSearchSource(type='url', url='https://www.nownews.com/news/6417733?utm_source=openai'),
 ActionSearchSource(type='url', url='https://hypebeast.com/zh/2024/7/hunter-x-hunter-manga-episode-38-release-info?utm_source=openai'),
 ActionSearchSource(type='url', url='https://www.4gamers.com.tw/news/detail/66089/hunter-x--hunter-new-comic?utm_source=openai'),
 ActionSearchSource(type='url', url='https://www.chilling.tw/article/89313?utm_source=openai'),
 ActionSearchSource(type='url', url='https://www.cool-style.com.tw/wd2/archives/1271926-%E5%86%A8%E6%A8%AB%E5%AE%A3%E5%B8%83%E3%80%8A%E7%8D%B5%E4%BA%BA%E3%80%8B420-%E8%A9%B1%E5%AE%8C%E6%88%90%EF%BC%8110-%E8%A9%B1%E8%A3%9C%E9%BD%8A%EF%BC%8C%E6%9C%89%E6%9C%9B%E5%9C%A8%E8%BF%91%E6%9C%9F?utm_source=openai'),
 ActionSearchSource(type='url', url='htt

In [69]:
print(response.output_text)

截至2026年3月28日，關於《獵人x獵人》（Hunter × Hunter）的最新進度，以下是近期的更新：

- **2026年1月14日**：作者冨樫義博在X（前稱Twitter）上宣布，第414話原稿已完成，並附上庫拉皮卡、西索等角色的插圖，令粉絲興奮不已。 ([nownews.com](https://www.nownews.com/news/6775626?utm_source=openai))

- **2024年12月17日**：冨樫義博表示，正在進行未來50話的台詞和時序確認與調整，並公開了部分登場角色的資料。 ([oricon.co.jp](https://www.oricon.co.jp/news/2359490/full/?utm_source=openai))

- **2024年7月22日**：集英社官方宣布，《獵人x獵人》第38卷將於2024年9月4日發售，距離上一卷發行已近兩年。 ([4gamers.com.tw](https://www.4gamers.com.tw/news/detail/66089/hunter-x--hunter-new-comic?utm_source=openai))

綜合以上資訊，冨樫義博近期的工作進度顯示，《獵人x獵人》的連載正在穩步推進，粉絲們可期待未來更多內容的發布。 


#### 使用者位置（User location）

為了依據地理位置來精細化搜尋結果，你可以使用國家、城市、地區及／或時區來指定使用者的大致位置。

- **城市（city）**與**地區（region）**欄位為自由文字字串，例如：`Minneapolis` 與 `Minnesota`。
- **國家（country）**欄位為兩位字母的 ISO 國家代碼，例如：`US`。（ISO 3166-1 alpha-2）
- **時區（timezone）**欄位為 IANA 時區格式，例如：`America/Chicago`。


In [70]:
response = client.responses.create(
         model="gpt-4o-mini",
         tools=[{"type": "web_search",
                 "user_location": {
                    "type": "approximate",
                    "country": "US",
                     },
                 "search_context_size": "medium"
                }],
         include=["web_search_call.action.sources"],
         input="幫我查詢HunterXHunter 最新的進度"
    )
                
print(response.output_text)

《獵人x獵人》（Hunter × Hunter）自1998年連載以來，因作者冨樫義博的健康狀況，連載進度時有中斷。然而，近期有關連載進度的更新值得關注。

2024年12月17日，冨樫義博在X（前稱Twitter）上更新，表示正在進行未來50話的台詞和時序確認與調整，並計劃在完成後重新開始原稿的上色等工作。 ([oricon.co.jp](https://www.oricon.co.jp/news/2359490/full/?utm_source=openai))

2025年8月20日，他再次更新，表示第413話的背景已完成，並附上原稿照片，顯示連載進度有所推進。 ([chilling.tw](https://www.chilling.tw/article/89313?utm_source=openai))

2026年1月14日，冨樫義博在X上宣布第414話原稿完成，並附上庫拉皮卡、西索等角色的插圖，顯示連載進度持續提升。 ([nownews.com](https://www.nownews.com/news/6775626?utm_source=openai))

此外，2024年9月4日，單行本第38卷正式發售，距離上一卷發售已近兩年，顯示連載進度有所恢復。 ([hypebeast.com](https://hypebeast.com/zh/2024/7/hunter-x-hunter-manga-episode-38-release-info?utm_source=openai))

綜合以上資訊，雖然連載進度仍不穩定，但冨樫義博近期的更新顯示連載有所推進，粉絲可期待未來更多內容的發布。 


#### 其他參數（Other arguments）

- **網域過濾（Domain filtering）**（僅適用於 gpt-5 與 o-series 模型）
- **推理（reasoning）**（僅適用於 gpt-5 與 o-series 模型）
    - **effort（推理強度）**：
        - ***minimal***：最小
        - ***low***：低
        - ***medium***：中
        - ***high***：高
    - **summary（摘要程度）**：
        - ***auto***：自動
        - ***concise***：精簡
        - ***detailed***：詳細  

- **tool_choice（工具選擇）**
    - ***none***：表示模型不會呼叫任何工具，而是直接產生訊息。
    - ***auto***：表示模型可在產生訊息或呼叫一個或多個工具之間自行選擇。
    - ***required***：表示模型必須呼叫一個或多個工具。

In [71]:
response = client.responses.create(
  model="gpt-5",
  reasoning={"effort": "low"},
  tools=[
      {
          "type": "web_search",
          "filters": {
              "allowed_domains": [
                  "pubmed.ncbi.nlm.nih.gov",
                  "clinicaltrials.gov",
                  "www.who.int",
                  "www.cdc.gov",
                  "www.fda.gov",
              ]
          },
      }
  ],
  tool_choice="auto",
  include=["web_search_call.action.sources"],
  input="請進行網路搜尋，了解 semaglutide 在治療糖尿病中的使用方式。用繁體中文回應",
)


In [72]:
print(response.output_text)

以下內容依據美國 FDA、PubMed 與 ClinicalTrials.gov 的權威資料彙整（查詢日期：2026-03-28）。

概述
- Semaglutide 是一種 GLP-1 受體致效劑，用於成人第二型糖尿病，能改善血糖控制；美國核准的糖尿病適應症產品包含每週一次皮下注射的 Ozempic 與每日一次口服錠的 Rybelsus（另有 Wegovy 為體重管理用途）。([fda.gov](https://www.fda.gov/drugs/postmarket-drug-safety-information-patients-and-providers/medicamentos-que-contienen-semaglutida-comercializados-para-la-diabetes-de-tipo-2-o-la-perdida-de?utm_source=openai))
- 作用機制屬葡萄糖依賴性促進胰島素分泌、抑制升糖素、並具體重下降效益的藥物類別。([fda.gov](https://www.fda.gov/drugs/postmarket-drug-safety-information-patients-and-providers/medicamentos-que-contienen-semaglutida-comercializados-para-la-diabetes-de-tipo-2-o-la-perdida-de?utm_source=openai))

療效與臨床證據（第二型糖尿病）
- 皮下注射：SUSTAIN‑6 心血管結局試驗以每週 0.5 mg 或 1.0 mg 劑量顯示，相較安慰劑可降低三項主要不良心血管事件（MACE）的風險；同時觀察到糖化血色素下降與體重減輕。([pubmed.ncbi.nlm.nih.gov](https://pubmed.ncbi.nlm.nih.gov/27633186/?utm_source=openai))
- 口服：PIONEER‑6 顯示口服 semaglutide（逐步至最多 14 mg 每日）符合心血管安全性，並可改善血糖與體重。([pubmed.ncbi.nlm.nih.gov](https://pubmed.ncbi.nlm.nih.gov/31185157/?utm_s

### 建立websearch工具

In [73]:
@tool
def websearh_tool(query: str) -> str:
    # 工具需要一個docstring，做為引導LLM哪個時候該使用這個工具
    """Use this tool to find the latest information or information you are not sure"""

    response = client.responses.create(
                    model="gpt-4o-mini",
                    tools=[
                        {"type": "web_search",}
                    ],
                    tool_choice="auto",
                    input=query)
    
    return response.output_text

In [74]:
llm = ChatOpenAI(model="gpt-4o-mini")

llm_with_tools = llm.bind_tools([websearh_tool])

# Run
resp = llm_with_tools.invoke("台灣2024總統大選結果")

Trace(trace_id=tr-437fde489c4ea95c83825c5ea185deb0)

In [75]:
resp

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 64, 'total_tokens': 89, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DOI8TD0wsQOlP4OfF05S9yYWzni0A', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--8f530fec-338c-4189-b4df-9aeadacfed47-0', tool_calls=[{'name': 'websearh_tool', 'args': {'query': '台灣2024總統大選結果'}, 'id': 'call_5LxIv7ni6eTR2HBY6Sp7QTbw', 'type': 'tool_call'}], usage_metadata={'input_tokens': 64, 'output_tokens': 25, 'total_tokens': 89, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [76]:
tool_call = resp.tool_calls[0]
result = eval(tool_call['name']).invoke(tool_call['args'])

print(result)

在2024年1月13日舉行的中華民國第16任總統副總統選舉中，民進黨候選人賴清德與蕭美琴成功當選，獲得5,586,019票，得票率為40.05%。 ([cna.com.tw](https://www.cna.com.tw/news/aipl/202401135011.aspx?utm_source=openai))國民黨候選人侯友宜與趙少康獲得4,671,021票，得票率為33.49%；民眾黨候選人柯文哲與吳欣盈則獲得3,690,466票，得票率為26.46%。 ([neww.tw](https://neww.tw/2024-result/?utm_source=openai))

在立法委員選舉方面，國民黨獲得52席，成為最大黨；民進黨獲得51席，民眾黨獲得8席，無黨籍及未經政黨推薦的候選人獲得2席。 ([cna.com.tw](https://www.cna.com.tw/topic/newstopic/4354.aspx?utm_source=openai))這使得立法院呈現「三黨不過半」的局面，未來立法院的議程和決議可能需要各黨協商合作。

賴清德在當選後表示，未來將不分黨派、用人唯才，強調維持台海和平穩定是他的重要使命，並將以對話取代對抗。 ([news.pts.org.tw](https://news.pts.org.tw/article/676204?utm_source=openai))

若您想觀看2024年台灣大選的開票特別報導，以下是公視的直播影片：

[2024台灣大選開票特別報導](https://www.youtube.com/live/hwy2kNslIiU?utm_source=openai)
 


Trace(trace_id=tr-35f465521d6f1169a40401ed05b0307b)

## 呼叫複數Tools

In [77]:
from openai import OpenAI

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])


@tool
def websearh_tool(query: str) -> str:
    """
    使用網路搜尋取得最新、即時或不在模型既有知識範圍內的資訊。

    此工具應在以下情況下由 LLM 調用：
    1. 問題涉及時事或近期事件，例如：
       - 當前或最近的政治情勢、政策變化、選舉結果
       - 明星、網紅的最新八卦或動態
       - 體育賽事結果、戰績、轉會消息
       - 任何具有時間敏感性、會隨時變動的資訊
    2. 當 LLM 無法確定自身知識是否足以準確回答問題時，
       特別是可能超出模型知識截止時間或需要即時驗證的內容。

    使用原則：
    - 若問題可由一般常識、穩定不變的知識或訓練資料中可靠回答，則不應調用。
    - 若答案的正確性高度依賴最新資訊，應優先調用此工具。
    - 搜尋關鍵字應簡潔且包含必要的上下文，以提高結果相關性。

    參數：
        query (str): 用於搜尋引擎的關鍵字或自然語言查詢。

    回傳：
        str: 來自網路搜尋結果的摘要或相關資訊，用於輔助生成最終回答。

    範例：
        websearh_tool("2025 台灣 總統 最新 民調")
        websearh_tool("今天 NBA 勇士 比賽 結果")
        websearh_tool("某某明星 最近 發生 什麼事")
    """
    response = client.responses.create(
        model="gpt-4o-mini",
        tools=[
            {"type": "web_search"}
        ],
        tool_choice="auto",
        input=query
    )

    return response.output_text

In [78]:
@tool
def mathematic_tool(expression: str) -> str:
    """
    執行數學運算，適用於需要精確計算的問題。

    參數：
        expression (str): 要計算的數學表達式，例如 "2 + 3 * (4 - 1)"

    回傳：
        str: 計算結果。
    """
    try:
        result = eval(expression, {"__builtins__": {}})
        return str(result)
    except Exception as e:
        return f"Error evaluating expression: {e}"


In [79]:
model = ChatOpenAI(
    openai_api_key=os.environ['OPENAI_API_KEY'],
    model_name="gpt-4o",
    temperature=0,
    name='llm_model_with_tool'
).bind_tools([websearh_tool, mathematic_tool])

In [80]:
response = model.invoke(
    "What are the latest NBA scores and what is 23 * 17?"
)

print(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 504, 'total_tokens': 558, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_9894c391cd', 'id': 'chatcmpl-DOIBIqvRHcEucY5WyOKbVFaLFOawc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--e1dccadb-60fc-4e18-bda4-0f269f2161fb-0' tool_calls=[{'name': 'websearh_tool', 'args': {'query': 'NBA latest scores'}, 'id': 'call_3uwPYK8ZwqWSwcLhJAB7WWgi', 'type': 'tool_call'}, {'name': 'mathematic_tool', 'args': {'expression': '23 * 17'}, 'id': 'call_3wL2p8HXd9rwLHRg4aUmtGJB', 'type': 'tool_call'}] usage_metadata={'input_tokens': 504, 'output_tokens': 54, 'total_tokens': 558, 'input_token_details': {

Trace(trace_id=tr-cba4f8b001515baa36235ba3c17a4852)

In [81]:
response.tool_calls

[{'name': 'websearh_tool',
  'args': {'query': 'NBA latest scores'},
  'id': 'call_3uwPYK8ZwqWSwcLhJAB7WWgi',
  'type': 'tool_call'},
 {'name': 'mathematic_tool',
  'args': {'expression': '23 * 17'},
  'id': 'call_3wL2p8HXd9rwLHRg4aUmtGJB',
  'type': 'tool_call'}]

### 結合 AIMessage和Tool Calls產生最後的結果

In [82]:
from langchain_core.messages import ToolMessage

messages = [response]

for tool_call in response.tool_calls:
    result = eval(tool_call['name']).invoke(tool_call['args'])
    tool_msg = ToolMessage(
        content=str(result),          # usually a string or simple text
        tool_call_id=tool_call['id']    # must match the AIMessage tool_call id
    )
    messages.append(tool_msg)

model.invoke(messages)

AIMessage(content='Here are the latest NBA scores from Friday, March 27, 2026:\n\n- **Clippers (LAC) @ Pacers (IND)**: LAC 114 - IND 113\n- **Hawks (ATL) @ Celtics (BOS)**: ATL 102 - BOS 109\n- **Heat (MIA) @ Cavaliers (CLE)**: MIA 128 - CLE 149\n- **Bulls (CHI) @ Thunder (OKC)**: CHI 113 - OKC 131\n- **Rockets (HOU) @ Grizzlies (MEM)**: HOU 119 - MEM 109\n- **Pelicans (NOP) @ Raptors (TOR)**: NOP 106 - TOR 119\n- **Jazz (UTA) @ Nuggets (DEN)**: UTA 129 - DEN 135\n- **Wizards (WAS) @ Warriors (GSW)**: WAS 126 - GSW 131\n- **Mavericks (DAL) @ Trail Blazers (POR)**: DAL 100 - POR 93\n- **Nets (BKN) @ Lakers (LAL)**: BKN 99 - LAL 116\n\nFor upcoming games and more detailed statistics, you can visit the official NBA website.\n\nAdditionally, the result of the mathematical expression \\(23 \\times 17\\) is **391**.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 288, 'prompt_tokens': 1152, 'total_tokens': 1440, 'completion_tokens_details': {'ac

[Trace(trace_id=tr-aef22de781aaac66bb8835c344f44099), Trace(trace_id=tr-3f93fdda0501fc58c4f3eecc941e544a), Trace(trace_id=tr-10a75f322d892c84a1e31511bc839350)]

## 工具調用模組化

總結上面的工具調用經驗，將工具調用，信息(messages)整合，包裝在同一個function裡。

In [83]:
def follow_up_answer(aimessage):

    messages = [aimessage]

    for tool_call in aimessage.tool_calls:
        result = eval(tool_call['name']).invoke(tool_call['args'])
        tool_msg = ToolMessage(
            content=str(result),          # usually a string or simple text
            tool_call_id=tool_call['id']    # must match the AIMessage tool_call id
        )
        messages.append(tool_msg)
    
    return model.invoke(messages)

# follow_up_answer(aimessage=response)

## ChatBot 本體

### LLM 沒有記憶性

In [84]:
from langchain_core.messages import HumanMessage, AIMessage

model = ChatOpenAI(model="gpt-4o-mini")

message = HumanMessage(
            content="Translate this sentence from English to Chinese (繁體中文): I love programming."
        )

model.invoke(
    [message]
)

AIMessage(content='我愛程式設計。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 23, 'total_tokens': 30, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_648a9c6b83', 'id': 'chatcmpl-DOIGHXs8d6ndnuFbpdtdhohthNxSw', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--5739fb30-0bad-4dff-a8fc-b1d92610dc56-0', usage_metadata={'input_tokens': 23, 'output_tokens': 7, 'total_tokens': 30, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Trace(trace_id=tr-bf04f00d3b192a029ae1ecd08a5da6fb)

In [85]:
model.invoke([HumanMessage(content="What did you just say?")])

AIMessage(content='I mentioned that I have been trained on data up to October 2023. If you have a specific question or topic you’d like to discuss, feel free to let me know!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 13, 'total_tokens': 50, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_648a9c6b83', 'id': 'chatcmpl-DOIGX4gjfa760KCIjNRLh6aZ3Jis0', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--ed96bbd2-d1c5-4f50-8565-008441f56a29-0', usage_metadata={'input_tokens': 13, 'output_tokens': 37, 'total_tokens': 50, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Trace(trace_id=tr-041ec137a65cf219bb58a766f03f8874)

### 外部記憶

如何將外部記憶加入?

1. 迭代聊天紀錄來做為外部記憶

In [86]:
model.invoke(
    [
        HumanMessage(
            content="Translate this sentence from English to  Chinese (繁體中文): I love programming."
        ),
        AIMessage(content="我愛程式設計."),
        HumanMessage(content="What did you just say?"),
    ]
)

AIMessage(content='I translated the sentence "I love programming" into Traditional Chinese as "我愛程式設計."', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 45, 'total_tokens': 66, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_648a9c6b83', 'id': 'chatcmpl-DOIHD3VfcgtH0CWyXxiHpdMkAjLbT', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--cb63dacb-8f88-4b9e-ad16-93f199a7f3af-0', usage_metadata={'input_tokens': 45, 'output_tokens': 21, 'total_tokens': 66, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Trace(trace_id=tr-8fac958691c12cc22534409c57453960)

2. 透過 MessagePlaceholder接收外部記憶

In [87]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder, SystemMessagePromptTemplate

system_prompt = PromptTemplate(template=("You are a helpful assistant. Answer all questions to the best of your "
                                         "ability."))
system_message = SystemMessagePromptTemplate(prompt=system_prompt)

prompt = ChatPromptTemplate.from_messages(
    [
        system_message,
        MessagesPlaceholder(variable_name="messages"),
    ]
)

### 建立邏輯鍊條

In [88]:
pipeline_ = prompt | model

In [89]:
from langchain_core.messages import HumanMessage, AIMessage

pipeline_.invoke(
    {
        "messages": [
            HumanMessage(
            content="Translate this sentence from English to Chinese (繁體中文): I love programming."
            ),
            AIMessage(content="我愛程式設計."),
            HumanMessage(content="What did you just say?"),
            ],
    }
)

AIMessage(content='I translated "I love programming" into Traditional Chinese as "我愛程式設計."', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 64, 'total_tokens': 83, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_648a9c6b83', 'id': 'chatcmpl-DOIKDYISEDP3N0nrAao1Ew3PnzJlp', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--5418d6d7-4cc3-45cc-abdd-68487e1c3a49-0', usage_metadata={'input_tokens': 64, 'output_tokens': 19, 'total_tokens': 83, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Trace(trace_id=tr-6971ecff365da9df691f74a6655d6b04)

### ChatBot 最小範例

結構上就是一個while loop

In [90]:
from langchain_core.output_parsers import StrOutputParser

messages = []
pipeline_ = prompt|model|StrOutputParser()

while True:
    question = input("What do you want to ask: ")
    if question == "QUIT":
        break
    messages.append(HumanMessage(content=question))
    response = pipeline_.invoke({"messages": messages})

    print(response)
    messages.append(AIMessage(content=response))
    

What do you want to ask:  假設今天是一月一日，後天是幾號?


如果今天是一月一日，後天就是一月三日。


What do you want to ask:  那一個月後是幾月?


如果今天是一月一日，一個月後是二月一日。


What do you want to ask:  QUIT


[Trace(trace_id=tr-63e7dda03e1320109e6a251b590b40d3), Trace(trace_id=tr-d67ef2fb7f7f4d1fcd020531253abcd8)]

## ChatBot + 檢索系統整合

🎯 **本章學完你將能學會什麼：**

- 理解如何結合 FAISS 向量資料庫與 ChatBot  
- 學會建立可檢索文本的 RAG（Retrieval-Augmented Generation）工作流  
- 掌握 StructuredTool 與 Retriever 工具的實際應用  
- 能讓 ChatBot 根據知識庫回答特定問題（如唐詩檢索）  

📘 **最終你將具備的能力：**  
能夠構建一個具備「外部記憶與知識檢索」能力的智慧 ChatBot，結合語意搜尋與生成。  

In [91]:
from textwrap import dedent

from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.runnables import ConfigurableField
from langchain_core.tools import tool
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder, SystemMessagePromptTemplate, HumanMessagePromptTemplate

from src.io.path_definition import get_project_dir
from initialization import credential_init


credential_init()

# 引入唐詩向量數據庫
filename = os.path.join(get_project_dir(), "tutorial", "week_2", "poem_faiss_index")

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

vectorstore = FAISS.load_local(
    filename, embeddings, allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(seearch_type='similarity').configurable_fields(\
        search_kwargs=ConfigurableField(id="search_kwargs")
    )


class PoemRetrieverArgs(BaseModel):
    query: str = Field(description="The keyword or phrase to search for Tang poems. 用來搜尋唐詩的關鍵字或是句子")
    k: int = Field(1, description="The number of poems to retrieve.")


def _poem_retriever(query: str, k: int):
    output = retriever.invoke(query, config={"configurable": {"search_kwargs": {"k": k}}})
    return output

C:\Users\Ling\AppData\Local\Temp\ipykernel_17616\3384218770.py:21: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 056c21f3-0fa1-4983-86a7-456ed35fcd1e)')' thrown while requesting HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 3df7e144-7529-4071-8d85-6d9e9ddd036d)')' thrown while requesting HEAD https://huggingface.co/BAAI/bg

In [92]:
@tool
def poem_retriever_tool(input_: PoemRetrieverArgs):
    """使用這個工具來搜尋唐詩; Use this tool to search for Tang poems."""
    return _poem_retriever(query=input_.query, k=input_.k)

In [93]:
from langchain_core.messages import SystemMessage
# from langchain_google_genai import ChatGoogleGenerativeAI

system_message = SystemMessage(content=dedent("""
You are a helpful assistant. 
Answer all questions to the best of your ability.
"""))

prompt = ChatPromptTemplate.from_messages(
    [
        system_message,
        MessagesPlaceholder(variable_name="messages"),
    ]
)

model = ChatOpenAI(openai_api_key=os.environ['OPENAI_API_KEY'],
                   model_name="gpt-4o-2024-05-13", temperature=0)

model_with_tools = model.bind_tools([poem_retriever_tool])

chatbot_pipeline = prompt | model_with_tools

基本測試

In [94]:
messages = []

question = "幫我找3首關於對於人生感嘆的唐詩"

messages.append(HumanMessage(content=question))

output = chatbot_pipeline.invoke({"messages": messages})

Trace(trace_id=tr-2acc32fc3ee6511ed1aa460fcca2d150)

In [95]:
output

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 142, 'total_tokens': 170, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-05-13', 'system_fingerprint': 'fp_7fdf9aa25b', 'id': 'chatcmpl-DOIQAG2fvxjMBe62KdfZICVZiWeIg', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--7677cbff-5f7d-430e-9fc8-124327beb1fd-0', tool_calls=[{'name': 'poem_retriever_tool', 'args': {'input_': {'query': '人生感嘆', 'k': 3}}, 'id': 'call_NAIs5KqooLNJNLMapXHUgEfQ', 'type': 'tool_call'}], usage_metadata={'input_tokens': 142, 'output_tokens': 28, 'total_tokens': 170, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [96]:
output = follow_up_answer(aimessage=output)

[Trace(trace_id=tr-1bccc064dcfcba6221e9b7fd8b4a7625), Trace(trace_id=tr-905b52f9e617dac2e5902b069e0c5eb7)]

In [97]:
output

AIMessage(content='以下是三首與「人生感嘆」相關的詩：\n\n1. **《登幽州臺歌》** - 陳子昂\n   ```\n   前不見古人，後不見來者。\n   念天地之悠悠，獨愴然而涕下。\n   ```\n\n2. **《夢李白二首之二》** - 杜甫\n   ```\n   浮雲終日行，遊子久不至。\n   三夜頻夢君，情親見君意。\n   告歸常局促，苦道來不易。\n   江湖多風波，舟楫恐失墜。\n   出門搔白首，若負平生志。\n   冠蓋滿京華，斯人獨憔悴。\n   孰云網恢恢，將老身反累。\n   千秋萬歲名，寂寞身後事。\n   ```\n\n3. **《江州重別薛六柳八二員外》** - 劉長卿\n   ```\n   生涯豈料承優詔，世事空知學醉歌。\n   江上月明胡雁過，淮南木落楚山多。\n   寄身且喜滄洲近，顧影無如白髮何。\n   今日龍鍾人共老，愧君猶遣慎風波。\n   ```\n\n這些詩句表達了對人生無常、時光流逝以及個人理想未竟的感嘆。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 348, 'prompt_tokens': 458, 'total_tokens': 806, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-05-13', 'system_fingerprint': 'fp_7fdf9aa25b', 'id': 'chatcmpl-DOIQVZRUBAnAxpqAkeiqgyfa0QleS', 'service_tier': 'default', 'finish_reason': 'sto

In [98]:
messages = []

while True:
    question = input("What do you want to ask: ")
    if question == "QUIT":
        break
    messages.append(HumanMessage(content=question))
    output = chatbot_pipeline.invoke({"messages": messages})

    # 若是有調用工具的話
    if output.tool_calls != []:
        response = follow_up_answer(output).content
    else:
        response = output.content

    print("***********************")
    print(response)
    print("***********************")
    
    messages.append(AIMessage(content=response))

What do you want to ask:  1 + 1 是多少


***********************
1 + 1 等于 2。
***********************


What do you want to ask:  黃河入海流的整首內容


***********************
這句詩出自唐代詩人王之渙的《登鸛鵲樓》。全詩如下：

```
白日依山盡，
黃河入海流。
欲窮千里目，
更上一層樓。
```

這首詩描寫了詩人登上鸛鵲樓時所見的壯麗景色，並表達了詩人追求更高目標的志向。
***********************


What do you want to ask:  QUIT


[Trace(trace_id=tr-bfac851406762147d26c4da641f9c33a), Trace(trace_id=tr-a80e86e8e670edd55c262102212312c0), Trace(trace_id=tr-66fed33966d62c67eeba1798153421fe), Trace(trace_id=tr-d26bccc648e4a38d670b1cdc0943b387)]

### 使用代碼解決數學問題工具

- OpenAI API: https://platform.openai.com/docs/guides/tools-code-interpreter
- 自己玩玩看。但我們也可以自己手搓寫代碼的工具

1. 代碼產生的邏輯鍊條

In [99]:
import re
from textwrap import dedent

from langchain_openai import ChatOpenAI
from langchain_core.runnables import chain
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
from langchain_core.tools import StructuredTool
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from pydantic import BaseModel, Field

from initialization import credential_init

credential_init()


def build_standard_chat_prompt_template(kwargs):

    messages = []
 
    if 'system' in kwargs:
        content = kwargs.get('system')
        prompt = PromptTemplate(**content)
        message = SystemMessagePromptTemplate(prompt=prompt)
        messages.append(message)  

    if 'human' in kwargs:
        content = kwargs.get('human')
        prompt = PromptTemplate(**content)
        message = HumanMessagePromptTemplate(prompt=prompt)
        messages.append(message)
        
    chat_prompt = ChatPromptTemplate.from_messages(messages)
    
    return chat_prompt


# 調用第二周的內容
@chain
def code_execution(code):

    match = re.findall(r"python\n(.*?)\n```", code, re.DOTALL)
    python_code = match[0]
    
    lines = python_code.strip()

    globals_dict = {}
    locals_dict = {}
    
    # ⚠️ globals 和 locals 用同一個
    exec(lines, globals_dict, globals_dict)
    
    return globals_dict.get("answer")


system_template = dedent("""
    You are a highly skilled Python developer. Your task is to generate Python code strictly based on the user's instructions.
    Leverage statistical and mathematical libraries such as `statsmodels`, `scipy`, and `numpy` where appropriate to solve the problem.
    Your response must contain only the Python code — no explanations, comments, or additional text.
    Always copy the final answer to a variable `answer`
    """)

human_template = dedent("""
                        {query}
                        Always copy the final answer to a variable `answer`
                        Code:
                        """)


input_ = {"system": {"template": system_template},
          "human": {"template": human_template,
                    "input_variable": ["query"]}}


model = ChatOpenAI(model="gpt-4o-mini")

chat_prompt_template = build_standard_chat_prompt_template(input_)

code_generation = chat_prompt_template|model|StrOutputParser()

code_pipeline = code_generation|code_execution

In [100]:
code_pipeline.invoke("Calculate the area of a circle with radius 8.3125")

217.07668925527284

Trace(trace_id=tr-e0e82a10fb3e4433b3e082607e2d384e)

建立對應的工具

In [101]:
class CodeArgs(BaseModel):
    query: str = Field(description="User request; 用戶需求")


def _calculator(query: str,):
    output = code_pipeline.invoke(query)
    return output


@tool
def mathematic_tool(input_: CodeArgs):
    """
    Use this tool to solve mathematic related problem; 使用這個工具解決數學問題
    """

    return _calculator(query=input_.query)

In [102]:
from langchain_core.prompts import MessagesPlaceholder

model_with_tools = model.bind_tools([mathematic_tool])

system_prompt = PromptTemplate(template=dedent("""You are a helpful assistant. 
Answer all questions to the best of your ability.
"""))

system_message = SystemMessagePromptTemplate(prompt=system_prompt)

prompt = ChatPromptTemplate.from_messages(
    [
        system_message,
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chatbot_pipeline = prompt | model_with_tools

### 挑戰: 北一女段考考題

https://drive.google.com/file/d/1csHdgvc5WtbJZ4n39eozogVVPIkPWABf/view

- Q1: 以下敘述是否正確: 滿足方程式 x^2 + y^2 + 2x −10y + 30 = 0 之點(x, y)的圖形是一個圓
- Q2: 以下敘述是否正確: 過三點 A( 1, − 3), B( 2, 6 ), C( 4, 24 )的圓恰有一個
- Q3: 以下敘述是否正確: 直線 3x −4y + 7 = 0 與圓 (x − 2)^2 + (y + 3)^2 = 5 恰有一交點
- Q4: 以下敘述是否正確: 圓(x − 2)^2 + (y + 3)^2 = 5 上恰有二點與直線 3x −4y −13= 0 的距離等於 2
- Q5: 以下敘述是否正確: P(a, b) 為 圓 (x − 2)^2 + ( y + 3)^2 = 4 上的點,則使 (a^2 + b^2)^0.5 為整數的點共有 8 個

### 物理奧林匹亞 台灣2025初試

某生在 1 atm，室溫（20°C）下，將少量的水倒入一壓力鍋中，然後將鍋蓋封緊密閉，並將其置於一爐上加熱。在鍋內的水完全汽化為水蒸氣時，該生測得鍋內的溫度
T = 115°C，壓力為 3 atm。試估計原先倒入的水與壓力鍋內部的體積比值為何？
（10）

（已知：空氣和水的分子量分別為 28.8 g/mol 和 18.0 g/mol；
1 atm = 1.01 × 10^5 N/m²；
氣體常數 R = 8.31 J/mol·K。）

In [103]:
def follow_up_answer(aimessage):

    messages = [aimessage]

    for tool_call in aimessage.tool_calls:
        result = eval(tool_call['name']).invoke(tool_call['args'])
        tool_msg = ToolMessage(
            content=str(result),          # usually a string or simple text
            tool_call_id=tool_call['id']    # must match the AIMessage tool_call id
        )
        messages.append(tool_msg)
    
    return model.invoke(messages)

In [104]:
from langchain_core.messages import HumanMessage, AIMessage

messages = []

while True:
    question = input("Your question: ")
    if question == 'quit':
        break
    messages.append(HumanMessage(content=question))
    output = chatbot_pipeline.invoke({"messages": messages})
    
    try:
        if output.tool_calls != []:
            print("Call tool")
            final_answer = follow_up_answer(aimessage=output)
        else:
            print("No Tool")
            final_answer = output
        print(f"AI: {final_answer.content}")
        messages.append(AIMessage(content=final_answer.content))
    except KeyError:
        print(f"AI: {output.content}")
        messages.append(AIMessage(content=output.content))

Your question:  以下敘述是否正確: 滿足方程式 x^2 + y^2 + 2x −10y + 30 = 0 之點(x, y)的圖形是一個圓


Call tool
AI: The equation \( x^2 + y^2 + 2x - 10y + 30 = 0 \) does not represent a circle. A circle's equation typically takes the form \( (x - h)^2 + (y - k)^2 = r^2 \). 

To determine the type of conic section represented by this equation, we can rewrite it in standard form:

1. Rearrange the equation:
   \[
   x^2 + 2x + y^2 - 10y + 30 = 0
   \]

2. Complete the square for the \(x\) terms:
   \[
   x^2 + 2x = (x + 1)^2 - 1
   \]
   
3. Complete the square for the \(y\) terms:
   \[
   y^2 - 10y = (y - 5)^2 - 25
   \]

4. Substitute back into the equation:
   \[
   (x + 1)^2 - 1 + (y - 5)^2 - 25 + 30 = 0
   \]

5. Simplify:
   \[
   (x + 1)^2 + (y - 5)^2 + 4 = 0
   \]

This can be rearranged to highlight that the sum of squares is equal to a negative number:
\[
(x + 1)^2 + (y - 5)^2 = -4
\]

Since the left side (a sum of squares) cannot equal a negative number, this indicates that the equation does not represent a real circle, but rather an empty set. Thus, it does not correspond to

Your question:  以下敘述是否正確: 過三點 A( 1, − 3), B( 2, 6 ), C( 4, 24 )的圓恰有一個


No Tool
AI: 要判斷過三點 \( A(1, -3) \), \( B(2, 6) \), \( C(4, 24) \) 的圓是否恰有一個，我們可以檢查這三點是否共線。如果三點共線，則不能唯一決定一個圓；如果三點不共線，則可以唯一決定一個圓。

我們可以使用斜率來檢查這三點是否共線。計算點 \( A \) 和 \( B \) 的斜率，然後計算點 \( B \) 和 \( C \) 的斜率：

- 斜率 \( m_{AB} \) 由 \( A \) 到 \( B \)：
  \[
  m_{AB} = \frac{y_B - y_A}{x_B - x_A} = \frac{6 - (-3)}{2 - 1} = \frac{9}{1} = 9
  \]

- 斜率 \( m_{BC} \) 由 \( B \) 到 \( C \)：
  \[
  m_{BC} = \frac{y_C - y_B}{x_C - x_B} = \frac{24 - 6}{4 - 2} = \frac{18}{2} = 9
  \]

因為 \( m_{AB} = m_{BC} \)，因此三點 \( A \), \( B \), 和 \( C \) 共線。

由於這三點是共線的，所以過這三點的圓不存在唯一的圓。因此，敘述「過三點 \( A(1, -3) \), \( B(2, 6) \), \( C(4, 24) \) 的圓恰有一個」是不正確的。


Your question:  以下敘述是否正確: 直線 3x −4y + 7 = 0 與圓 (x − 2)^2 + (y + 3)^2 = 5 恰有一交點


...

Call tool
AI: The distance from the point (2, -3) to the line \( 3x - 4y + 7 = 0 \) is 5.0 units.


Your question:  QUIT


No Tool
AI: 如果有其他問題或需要進一步的幫助，隨時告訴我！有美好的一天！


Your question:  quit


[Trace(trace_id=tr-352bb43013f4fe16692f9a1f312a8a82), Trace(trace_id=tr-e3747e5073a2352fcf0304c8eaf423ca), Trace(trace_id=tr-eacd57bbcb316389ef020a028dec13f9), Trace(trace_id=tr-79d766c7edf92c2714e5ff7ffa4bbc1f), Trace(trace_id=tr-1b5dd1f0a0b138d2f04ec3eb13248702), Trace(trace_id=tr-36e0342d1fbdef3f5297f59b836b7572), Trace(trace_id=tr-afd83feda3960b1fc83d5408b284021e), Trace(trace_id=tr-50090d55ea358ad9b7a4ac90ce242ac7), Trace(trace_id=tr-1369e23a07b3e4eb2c1f26f435a7fc39), Trace(trace_id=tr-07a36efe6a67542e96d83859dfaa6acc)]

### 有辦法加入一些基本的機械學習來進行分析嗎?

我還不知道，應該是會蠻有趣的

到這裡你應該可以認識到，寫ChatBot本體並不困難，但一個ChatBot好不好用是由他所綑綁的工具決定。

剩下的就是UI

# Websearch 策略與資料來源設計

🎯 **本章學完你將能學會什麼：**

- 理解 Websearch 的優勢、限制與常見風險  
- 學會依據應用場景選擇正確的資訊來源（Wikipedia、Fandom、API）  
- 掌握資料品質控制與來源過濾策略  
- 掌握利用 LLM 動態生成 Pydantic Schema 的技巧  
- 能建立一個可從 Wikipedia / Fandom 抽取結構化資料的自動化 Pipeline  

📘 **最終你將具備的能力：**  
- 具備以 LLM 為核心的網路資訊擷取與分析能力，能建立高品質、多來源的 AI 研究與內容生成系統。
- 能夠設計結合爬蟲、LLM 代碼生成與結構化資料擷取的智慧資料分析系統。   

## Websearch 優缺點與應用策略

### 優點
- **多元化來源**：涵蓋範圍廣，能提供多角度資訊。
- **即時性**：能快速取得公開網頁上的最新內容。
- **靈活性**：適合需要「多來源比對」的問題。

### 缺點
- **碎片化**：資訊分散、格式不一，難以直接系統化使用。
- **品質參差不齊**：來源可靠度不同，可能存在錯誤或過時資訊。
- **限制與風險**：部分 API 或搜尋過程可能因政策、安全或授權而阻擋特定內容。

---

### 何時適合使用 Websearch
- 需要 **多角度觀點**（如新聞、論壇、社群資訊）。
- **開放探索**，對來源精確度要求不高。
- 無法透過單一可靠資料庫滿足需求時。

---

### 何時更適合使用特定來源
- **專門領域**：如 Wikipedia、Fandom Wiki（例如遊戲、小說、Warhammer 40k）。
- **結構化資料需求**：來源有規則的網址與內容組織，便於程式化檢索。
- **高可信度需求**：減少處理過多雜訊。

---

### API 使用注意事項
- **安全審查阻擋**：可能因涉及「不允許或敏感內容」而無法獲取公開資料。
- **授權與限制**：包含付費牆、Rate Limit、隱私規範等。
- **備援角色**：Websearch 可作為補充方案，但不一定是萬能解決方式。

---

### 總結
Websearch 提供了 **廣泛而多元的資訊**，但也帶來 **碎片化與品質問題**。  
若需求明確、可依靠結構化且可信的來源（如 Wikipedia、Fandom），應優先選用。  
若需要多角度、開放探索或無特定資料庫可依賴時，Websearch 才能發揮最大價值。


## 返無 歸一

- 假設你確定在某個來源肯定有資訊的時候，取得該來源的網址
- 使用第一周和第三周的技巧爬取網址的內容
- 透過LLM提取你要的訊息

In [105]:
from textwrap import dedent

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_openai import ChatOpenAI

from initialization import credential_init


def build_standard_chat_prompt_template(kwargs):

    messages = []
 
    if 'system' in kwargs:
        content = kwargs.get('system')
        prompt = PromptTemplate(**content)
        message = SystemMessagePromptTemplate(prompt=prompt)
        messages.append(message)  

    if 'human' in kwargs:
        content = kwargs.get('human')
        prompt = PromptTemplate(**content)
        message = HumanMessagePromptTemplate(prompt=prompt)
        messages.append(message)
        
    chat_prompt_template = ChatPromptTemplate.from_messages(messages)
    
    return chat_prompt_template

credential_init()

### 抽取Wikipedia的內容

In [106]:
import requests

session = requests.Session()

query = "獵人中的念能力系統"

# Wikipedia語言選擇
URL = "https://ja.wikipedia.org/w/api.php"


PARAMS = {
    "action": "parse",
    # Wikipedia 頁面的 關鍵字
    "page": "HUNTER×HUNTER",
    "prop": "text",
    "format": "json"
}

HEADERS = {
    "User-Agent": "AI Tutorial Bot/1.0 (mengchiehling@gmail.com)"
}

response = session.get(url=URL, params=PARAMS, headers=HEADERS)

data = response.json()

使用 bs4 處理數據 

In [107]:
from bs4 import BeautifulSoup

html_content = data['parse']['text']["*"]

soup = BeautifulSoup(html_content, "html.parser")

# 移除 style 和 script
for tag in soup(["style", "script"]):
    tag.decompose()

# 提取文字
text_content = soup.get_text(separator="\n")

# 清理空白與空行
cleaned_text = "\n".join(
    line.strip() for line in text_content.splitlines() if line.strip()
)

慢慢寫Pydantic物件是可以實現的目標的，但是在應用中我們希望有更好的自動化: 根據使用者的需求自動生成物件

我們嘗試結合代碼生成來輔助完成目標

In [114]:
code_example = dedent("""
    # Example 1: Name extraction using Pydantic and LangChain

    from pydantic import BaseModel, Field
    from langchain_core.output_parsers import PydanticOutputParser

    class NameExtractor(BaseModel):
        name: str = Field(description="The extracted name from the input text")

    output_parser = PydanticOutputParser(pydantic_object=NameExtractor)
    format_instructions = output_parser.get_format_instructions()

    ---
    # Example 2: Multiple product information extraction using Pydantic and Langchain

    from typing import List
    
    class Product(BaseModel):
        name: str = Field(description="Product")
        brand: str = Field(description="The brand name")
        country_code: str = Field(description="ISO 3166-1 alpha-2 of the country of the brand")

    class ProductOutput(BaseModel):
        products: List[Product] = Field(description="A list of products")

    output_parser = PydanticOutputParser(pydantic_object=NameExtractor)
    format_instructions = output_parser.get_format_instructions()
    
""")


system_template = dedent(f"""
    You are an expert Python developer specializing in large language models and the LangChain framework.
    Your objective is to generate **only valid, executable Python code** that solves the user's request.

    Requirements:
    - Use Pydantic models when defining structured outputs.
    - Ensure imports are correct and minimal.
    - Follow PEP 8 formatting standards.
    - Do not include any explanations, markdown, comments, or extra text outside the code block.
    - You have have the output_parser and the format_instruction of the Pydantic models.

    Example structure:
    {code_example}
""")

human_template = dedent("""
                        {query}
                        Code:
                        """)


input_ = {"system": {"template": system_template},
          "human": {"template": human_template,
                    "input_variable": ["query"]}}

chat_prompt_template = build_standard_chat_prompt_template(input_)

model = ChatOpenAI(model="gpt-4o-mini", name='parser code generator')

code_generation = chat_prompt_template|model|StrOutputParser()

In [115]:
generated_code = code_generation.invoke({"query": query})
print(generated_code)

```python
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class NenAbility(BaseModel):
    name: str = Field(description="The name of the Nen ability")
    category: str = Field(description="The category of the Nen ability")
    description: str = Field(description="A detailed description of the Nen ability")

class NenSystemOutput(BaseModel):
    abilities: list[NenAbility] = Field(description="A list of Nen abilities")

output_parser = PydanticOutputParser(pydantic_object=NenSystemOutput)
format_instructions = output_parser.get_format_instructions()
```


Trace(trace_id=tr-77c9e66ebbb9e746da249ff6f4dbe0e5)

### 代碼執行工具

In [116]:
import re

from langchain_core.runnables import chain

@chain
def code_execution(code):
    
    match = re.findall(r"python\n(.*?)\n```", code, re.DOTALL)
    python_code = match[0]
    
    lines = python_code.strip()

    globals_dict = {}
    locals_dict = {}
    
    # ⚠️ globals 和 locals 用同一個
    exec(lines, globals_dict, globals_dict)

    return globals_dict

執行產生的代碼

In [117]:
local_vars = code_execution.invoke(generated_code)

Trace(trace_id=tr-fe9f07093598ed31f29c5fbef5bc356e)

In [118]:
local_vars

{'__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, exceptions, and other objects.\n\nNoteworthy: None is the `nil' object; Ellipsis represents `...' in slices.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_class__>,
  '__import__': <function __import__>,
  'abs': <function abs(x, /)>,
  'all': <function all(iterable, /)>,
  'any': <function any(iterable, /)>,
  'ascii': <function ascii(obj, /)>,
  'bin': <function bin(number, /)>,
  'breakpoint': <function breakpoint>,
  'callable': <function callable(obj, /)>,
  'chr': <function chr(i, /)>,
  'compile': <function compile(source, filename, mode, flags=0, dont_inherit=False, optimize=-1, *, _feature_version=-1)>,
  'delattr': <function delattr(obj, name, /)>,
  'dir': <function dir>,
  'divmod': <function divmod(x, y, /)>,
  'eval': <

In [119]:
local_vars["__builtins__"] = {}

In [120]:
local_vars

{'__builtins__': {},
 'BaseModel': pydantic.main.BaseModel,
 'Field': <function pydantic.fields.Field(default: 'Any' = PydanticUndefined, *, default_factory: 'Callable[[], Any] | Callable[[dict[str, Any]], Any] | None' = PydanticUndefined, alias: 'str | None' = PydanticUndefined, alias_priority: 'int | None' = PydanticUndefined, validation_alias: 'str | AliasPath | AliasChoices | None' = PydanticUndefined, serialization_alias: 'str | None' = PydanticUndefined, title: 'str | None' = PydanticUndefined, field_title_generator: 'Callable[[str, FieldInfo], str] | None' = PydanticUndefined, description: 'str | None' = PydanticUndefined, examples: 'list[Any] | None' = PydanticUndefined, exclude: 'bool | None' = PydanticUndefined, exclude_if: 'Callable[[Any], bool] | None' = PydanticUndefined, discriminator: 'str | types.Discriminator | None' = PydanticUndefined, deprecated: 'Deprecated | str | bool | None' = PydanticUndefined, json_schema_extra: 'JsonDict | Callable[[JsonDict], None] | None' =

產生需要的pipeline

In [121]:
def build_answer_pipeline(output_parser, format_instructions):

    human_template = dedent("""{query}
                           
                           context:
                           {context}
                           output format instruction = {format_instruction} 
                        """)


    input_ = {"system": {"template": "You are a helpful AI assistant."},
              "human": {"template": human_template,
                        "input_variable": ["query"],
                        "partial_variables": {'format_instruction': format_instructions}
                        }}
    
    chat_prompt_template = build_standard_chat_prompt_template(input_)
    
    model = ChatOpenAI(model="gpt-4o-mini")
    
    answer_pipeline = chat_prompt_template|model|output_parser

    return answer_pipeline

執行

把用代碼產生的 output parser 和 format instructions 傳入

In [122]:
answer_pipeline = build_answer_pipeline(output_parser=local_vars['output_parser'], format_instructions=local_vars['format_instructions'])

output = answer_pipeline.invoke({"query": query, "context": cleaned_text})

Trace(trace_id=tr-d6b085b04796cfed1de06fc365a5e18a)

In [123]:
output

NenSystemOutput(abilities=[NenAbility(name='ジャジャン拳', category='強化系', description='ゴン＝フリークスの必殺技で、オーラをまとわせたパンチまたは攻撃。最も基本的な武闘スタイルを反映し、体力や攻撃力を最大限に引き出す。'), NenAbility(name='絶対時間（エンペラータイム）', category='具現化系', description='クラピカの能力。彼のクルタ族の緋の眼が発動すると、具現化した鎖が指定した念能力に対する防御力や捕縛力を増幅する。特定の条件が満たされた際には、戦闘中に彼の能力全般が強化される。'), NenAbility(name='バンジーガム', category='変化系', description='ヒソカの能力。オーラをガムのような性質に変え、自在に伸縮させることができる。相手を捕らえたり、道具として使用することができ、策略を駆使してバトルを展開する。'), NenAbility(name='一握りの火薬（リトルフラワー）', category='放出系', description='ゲンスルーの能力。オーラを爆薬に変換し、一定の条件下で爆発を引き起こす。爆発の威力を調整することで、反撃や攻撃手段に利用される。'), NenAbility(name='天上不知唯我独損（ハコワレ）', category='操作系', description='ナックルの能力。相手にオーラを与え、それが利息とともに増えていくことで、相手を絶の状態に強制的にする技。与えるオーラの量によって効果が変わる。')])

### Fan Wiki: Titan

萬機之神歐姆尼賽亞的化身

https://warhammer40k.fandom.com/wiki/Titan

In [124]:
import requests

from bs4 import BeautifulSoup


def parsing_process(url):
    """
    Fetches and extracts text content from a given URL.

    Parameters:
    url (str): The URL of the web page to fetch and parse.

    Returns:
    str: Cleaned text content extracted from the web page.

    Raises:
    requests.exceptions.RequestException: If an error occurs while fetching the URL.

    Notes:
    - This function sends a GET request to the specified URL.
    - It uses BeautifulSoup to parse the HTML content of the response.
    - Any <style> tags in the HTML are removed to extract only textual content.
    - The extracted text is cleaned by removing extra whitespace and empty lines.
    """
    headers = {
    "User-Agent": "Mozilla/5.0 (compatible; MyTest/1.0)",
    "Accept": "text/html,application/json"
    }
    
    # Send a GET request to the URL
    response = requests.get(url, headers=headers)

    # Get the content of the response
    html_content = response.text
    
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # 移除 style 和 script
    for tag in soup(["style", "script"]):
        tag.decompose()

    # Extract and print only the text content
    text_content = soup.get_text(separator='\n')

    # Clean up the text (optional)
    cleaned_text = '\n'.join(line.strip() for line in text_content.splitlines() if line.strip())
    
    return cleaned_text


url = "https://warhammer40k.fandom.com/wiki/Titan"

提取網頁內容

In [125]:
cleaned_text = parsing_process(url)

In [ ]:
# generated_code = code_generation.invoke({"query": query})

# local_vars = code_execution.invoke(generated_code)

建立新的pipeline並且執行

In [ ]:
# answer_pipeline = build_answer_pipeline(output_parser=local_vars['output_parser'], format_instructions=local_vars['format_instructions'])

# output = answer_pipeline.invoke({"query": query, "context": cleaned_text})

#### 試試看更具有挑戰的問題

In [126]:
query = "幫我找出所有陣營的所有泰坦級別"

generated_code = code_generation.invoke({"query": query})

Trace(trace_id=tr-4d5befd11bdb5fb0c4e77118c155c0ba)

In [127]:
local_vars = code_execution.invoke(generated_code)

answer_pipeline = build_answer_pipeline(output_parser=local_vars['output_parser'], format_instructions=local_vars['format_instructions'])

Trace(trace_id=tr-f2be393b86804ca19dae4bb7db754b5d)

In [128]:
output = answer_pipeline.invoke({"query": query, "context": cleaned_text})

Trace(trace_id=tr-5c1f6a7d364cae7008dca7eecd295dc5)

In [129]:
output

TitanOutput(titans=[Titan(faction='Imperium of Man', level='Rapier-class Titan'), Titan(faction='Imperium of Man', level='Warhound-class Titan'), Titan(faction='Imperium of Man', level='Dire Wolf-class Titan'), Titan(faction='Imperium of Man', level='Reaver-class Titan'), Titan(faction='Imperium of Man', level='Komodo-class Titan'), Titan(faction='Imperium of Man', level='Punisher-class Titan'), Titan(faction='Imperium of Man', level='Executor-class Titan'), Titan(faction='Imperium of Man', level='Apocalypse-class Titan'), Titan(faction='Imperium of Man', level='Carnivore-class Titan'), Titan(faction='Imperium of Man', level='Mirage-class Titan'), Titan(faction='Imperium of Man', level='Warbringer Nemesis-class Titan'), Titan(faction='Imperium of Man', level='Warlord-class Titan'), Titan(faction='Imperium of Man', level='Warmaster-class Titan'), Titan(faction='Imperium of Man', level='Emperor Titan'), Titan(faction='Imperium of Man', level='Castigator-class Titan'), Titan(faction='Chao

# 使用Streamlit 建立一個 Chatbot Demo

## 加入Callback 進行追蹤ChatBot

- pip install streamlit
- 執行 streamlit run tutorial/week_4/app_streamlit.py